In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.path import Path

In [ ]:
from scipy.interpolate import griddata
from scipy.spatial import cKDTree

In [ ]:
from mesa import Model
from mesa.datacollection import DataCollector
from mesa.space import MultiGrid
#from mesa.time import RandomActivation
from mesa.agent import Agent
import numpy as np
import random
import math
import random

In [ ]:
import csv

In [ ]:
import os
import rasterio
from matplotlib.animation import FuncAnimation
import glob, re
import seaborn as sns

In [ ]:
import imageio.v2 as imageio
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from collections import deque
from matplotlib.colors import BoundaryNorm, ListedColormap
from scipy.spatial import cKDTree

In [ ]:
from pyproj import Transformer, CRS

In [ ]:
from landlab import RasterModelGrid, imshow_grid
from landlab.components import FlowAccumulator

# Import and plot real DEM data

## Required user inputs

### Digital elevation map

In [ ]:
import os
import glob
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, calculate_default_transform
from rasterio.enums import Resampling

# -------------------------------
# Paths
# -------------------------------
path = r"C:\Users\jordan.kennedy\Documents\Project\MultiAgent simulation\Datasets\Rumney Marsh"
base = "RumneyMarsh"
file_name = f"{base}.csv"
csv_path = os.path.join(path, file_name)


def merge_with_rectified_inputs(filepaths, out_fp, resampling=Resampling.nearest):
    src_files = []
    temp_paths = []

    try:
        for fp in filepaths:
            src = rasterio.open(fp)
            if src.transform.is_rectilinear:
                src_files.append(src)
                temp_paths.append(None)
                continue

            transform, width, height = calculate_default_transform(
                src.crs, src.crs, src.width, src.height, *src.bounds
            )
            meta = src.meta.copy()
            meta.update({"transform": transform, "width": width, "height": height})
            tmp_fp = os.path.splitext(fp)[0] + "_rect.tif"

            with rasterio.open(tmp_fp, "w", **meta) as dst:
                for band_idx in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, band_idx),
                        destination=rasterio.band(dst, band_idx),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=src.crs,
                        resampling=resampling,
                    )

            src.close()
            src_files.append(rasterio.open(tmp_fp))
            temp_paths.append(tmp_fp)

        min_bands = min(src.count for src in src_files)
        mosaic, mosaic_transform = merge(src_files, indexes=list(range(1, min_bands + 1)))
        mosaic_meta = src_files[0].meta.copy()
        mosaic_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": mosaic_transform,
            "count": min_bands,
        })

        with rasterio.open(out_fp, "w", **mosaic_meta) as dest:
            dest.write(mosaic)
    finally:
        for src in src_files:
            src.close()
        for tmp_fp in temp_paths:
            if tmp_fp and os.path.exists(tmp_fp):
                os.remove(tmp_fp)


# Auto-detect and mosaic all .tif files in the directory
tif_files = glob.glob(os.path.join(path, "*.tif"))
if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {path}")

if len(tif_files) == 1:
    tif_path = tif_files[0]
    print(f"Single .tif found: {os.path.basename(tif_path)}")
else:
    print(f"Found {len(tif_files)} .tif files, mosaicking...")
    tif_path = os.path.join(path, f"{base}_mosaic.tif")
    merge_with_rectified_inputs(tif_files, tif_path)
    print(f"Mosaic written to: {tif_path}")


In [ ]:
tif_name = [os.path.basename(f) for f in tif_files]
print(tif_name)

In [ ]:
tif_paths = [os.path.join(path, f) for f in tif_name]
print(tif_paths)

In [ ]:
# -------------------------------
# Your bbox in lon/lat (EPSG:4326)
#Rumney Marsh ("EPSG:4326", "EPSG:26986")]

NE = (-70.997601, 42.441846)
SW = (-71.010142, 42.432882)
SE = (-70.997601, 42.432882)
NW = (-71.010142, 42.441846)




data_source = "NOAA"

In [ ]:
import rasterio


with rasterio.open(tif_path) as src:
    crs = src.crs
    DATA_CRS = crs
    DATA_EPSG = crs.to_epsg() if crs is not None else None
    CRS_LABEL = f"EPSG:{DATA_EPSG}" if DATA_EPSG is not None else str(crs)

    print("CRS:", crs)
    print("DATA_EPSG:", DATA_EPSG)
    print("CRS_LABEL:", CRS_LABEL)
    print("Width x Height:", src.width, "x", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Nodata:", src.nodata)


### Sattelite Imagery

#### Topo data

In [ ]:
import os
import glob
import rasterio
from rasterio.merge import merge
from rasterio.warp import reproject, calculate_default_transform
from rasterio.enums import Resampling

topo_path = rf"{path}\Topo"

# -----------------------------
# Normalize .tiff -> .tif
# -----------------------------
tiff_files = sorted(glob.glob(os.path.join(topo_path, "*.tiff")))

for src_fp in tiff_files:
    dst_fp = os.path.splitext(src_fp)[0] + ".tif"

    if os.path.exists(dst_fp):
        print(f"Already exists, skipping conversion: {os.path.basename(dst_fp)}")
        continue

    print(f"Converting {os.path.basename(src_fp)} -> {os.path.basename(dst_fp)}")

    with rasterio.open(src_fp) as src:
        meta = src.meta.copy()
        with rasterio.open(dst_fp, "w", **meta) as dst:
            dst.write(src.read())


def merge_with_rectified_inputs(filepaths, out_fp, resampling=Resampling.nearest):
    src_files = []
    temp_paths = []

    try:
        for fp in filepaths:
            src = rasterio.open(fp)
            if src.transform.is_rectilinear:
                src_files.append(src)
                temp_paths.append(None)
                continue

            transform, width, height = calculate_default_transform(
                src.crs, src.crs, src.width, src.height, *src.bounds
            )
            meta = src.meta.copy()
            meta.update({"transform": transform, "width": width, "height": height})
            tmp_fp = os.path.splitext(fp)[0] + "_rect.tif"

            with rasterio.open(tmp_fp, "w", **meta) as dst:
                for band_idx in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, band_idx),
                        destination=rasterio.band(dst, band_idx),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=src.crs,
                        resampling=resampling,
                    )

            src.close()
            src_files.append(rasterio.open(tmp_fp))
            temp_paths.append(tmp_fp)

        min_bands = min(src.count for src in src_files)
        mosaic, out_transform = merge(src_files, indexes=list(range(1, min_bands + 1)))

        out_meta = src_files[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "count": min_bands
        })

        with rasterio.open(out_fp, "w", **out_meta) as dest:
            dest.write(mosaic)
    finally:
        for src in src_files:
            src.close()
        for tmp_fp in temp_paths:
            if tmp_fp and os.path.exists(tmp_fp):
                os.remove(tmp_fp)


# -----------------------------
# Collect topo .tif files only
# -----------------------------
topo_files = sorted([
    fp for fp in glob.glob(os.path.join(topo_path, "*.tif"))
    if not os.path.basename(fp).lower().endswith("_cropped.tif")
    and "topo_mosaic" not in os.path.basename(fp).lower()
    and "_rect" not in os.path.basename(fp).lower()
])

if len(topo_files) == 0:
    raise FileNotFoundError(f"No .tif files found in: {topo_path}")

print(f"Found {len(topo_files)} topo tif file(s):")
for f in topo_files:
    print("  ", os.path.basename(f))

out_name = f"{file_name}_Topo_mosaic.tif"
topo_merged_path = os.path.join(topo_path, out_name)

if len(topo_files) == 1:
    topo_tif = topo_files[0]
    print("\nOnly one topo tif found; using it directly:")
    print(topo_tif)

else:
    merge_with_rectified_inputs(topo_files, topo_merged_path)
    topo_tif = topo_merged_path
    print("\nMerged topo mosaic saved to:")
    print(topo_tif)

print("\nActive topo_tif:")
print(topo_tif)


#### NAIP

In [ ]:
import os
import glob
import rasterio

naip_path = rf"{path}\NAIP"

tif_files = sorted([
    fp for fp in glob.glob(os.path.join(naip_path, "*.tif"))
    if not os.path.basename(fp).lower().endswith("_cropped.tif")
    and "naip_mosaic" not in os.path.basename(fp).lower()
    and "_rect" not in os.path.basename(fp).lower()
    and "_norm" not in os.path.basename(fp).lower()
    and "aligned_to_dem_grid" not in os.path.basename(fp).lower()
])

if len(tif_files) == 0:
    raise FileNotFoundError(f"No .tif files found in: {naip_path}")

print(f"Found {len(tif_files)} tif file(s):")
for f in tif_files:
    print("  ", os.path.basename(f))

naip_tif_files = tif_files.copy()

if len(tif_files) == 1:
    naip_fp = tif_files[0]
    print("\nOnly one tif found; using it directly:")
    print(naip_fp)
else:
    naip_fp = None
    print("\nMultiple NAIP tiles detected.")
    print("Deferring mosaic until the later DEM-grid alignment step so the merge can be clipped to the AOI.")

print("\nActive NAIP naip_fp:")
print(naip_fp)


### Specify 4 coordinates

In [ ]:

"""
#LytlePrairie

NE = (-120.251969,  44.273336)
SW = (-120.257686,  44.270931)
SE = (-120.251969,  44.270931)
NW = (-120.257686,  44.273336)

#Cambridge,VT

NE = (-72.810606,  44.633341)
SW = (-72.816405,  44.628992)
SE = (-72.810606,  44.628992)
NW = (-72.816405,  44.633341)


#Troy,VT

NE = (-72.385257,  45.006363)
SW = (-72.392559,  45.000573)
SE = (-72.385257,  45.000573)
NW = (-72.392559,  45.006363)

#Hineburgh

NE = (-73.114618,  44.320615)
SW = (-73.135033,  44.305049)
SE = (-73.114618,  44.305049)
NW = (-73.135033,  44.320615)


#Williston,VT

NE = (-73.062235, 44.448361)
SW = (-73.079434, 44.438056)
SE = (-73.062235, 44.438056)
NW = (-73.079434, 44.448361)


#Cambridge,VT

NE = (-72.810606,  44.633341)
SW = (-72.816405,  44.628992)
SE = (-72.810606,  44.628992)
NW = (-72.816405,  44.633341)


#BayouBeavers

NE = (-89.378824,  30.467509)
SW = (-89.387389,   30.460747)
SE = (-89.378824,   30.460747)
NW = (-89.387389,   30.467509)

#Minnesota (# 4326 (lon/lat deg) -> 6503 (Minnesota North, ftUS), "EPSG:4326", "EPSG:6503")
NW = (-90.050850, 47.992967)
SW = (-90.050884, 47.991889)  #fixed
SE = (-90.048032,  47.991838)
NE = (-90.048021,  47.992884)  #fixed

#Finland
NW = ( 25.126260,   61.204110)
SW = ( 25.126260,   61.200423)
SE = ( 25.134242,   61.200423)
NE = ( 25.134242,   61.204110)
#https://asiointi.maanmittauslaitos.fi/karttapaikka/tilausvahvistus  https://www.maanmittauslaitos.fi/en/maps-and-spatial-data/datasets-and-interfaces/product-descriptions/elevation-model-2-m?utm_source=chatgpt.com
#National Land Survey of Finland

#TierraDelFuego
NW = (-68.215604,  -54.714897)
SW = (-68.215604,  -54.718327)
SE = (-68.209308,  -54.718327)
NE = (-68.209308,  -54.714897)
#https://www.ign.gob.ar/NuestrasActividades/Geodesia/ModeloDigitalElevaciones/Mapa

#Chile

NW = (-68.628304,  -54.853394)
SW = (-68.628304,  -54.861448)
SE = (-68.590848,  -54.861448)
NE = (-68.590848,  -54.853394)

# https://www.geoportal.cl/geoportal/catalog/35427/DEM%20Alos%20Palsar%20Regi%C3%B3n%20de%20Magallanes%20y%20de%20la%20Ant%C3%A1rtica%20Chilena?utm_source=chatgpt.com
#https://www.geoportal.cl/geoportal/catalog/35432/DEM%20Alos%20Palsar%20Regi%C3%B3n%20de%20Los%20R%C3%ADos?utm_source=chatgpt.com

#AlaskaTundra
NW = (-162.507625,  66.867081)
SW = (-162.507625,  66.863122)
SE = (-162.494745,  66.863122)
NE = (-162.494745,  66.867081)

#Bamff national park
NW = (-115.627961,   51.178700)
SW = (-115.627961,   51.173824)
SE = (-115.612711,   51.173824)
NE = (-115.612711,   51.178700)

#Pasquia Hills 2
NW = (-102.328355,  53.419427)
SW = (-102.328355,  53.417733)
SE = (-102.325770,  53.417733)
NE = (-102.325770,  53.419427)

#Pasquia Hills
NW = (-102.324338,  53.417643)
SW = (-102.324338,  53.416542)
SE = (-102.322038,  53.416542)
NE = (-102.322038,  53.417643)

#Geer/Arnoux Coulee
NW = (-113.190634,  48.695426)
SW = (-113.190634,  48.689101)
SE = (-113.180253,  48.689101)
NE = (-113.180253,  48.695426)

#Pilling
NW = (-113.077074,  48.560456)
SW = (-113.077074,  48.556256)
SE = (-113.064159,  48.556256)
NE = (-113.064159,  48.560456)

#Sleeping Wolf
NW = (-113.058701,  48.564944)
SW = (-113.058701,  48.558888)
SE = (-113.047637,  48.558888)
NE = (-113.047637,  48.564944)

#Bitner 1
NW = (-112.765108,  48.660910)
SW = (-112.765108,  48.656106)
SE = (-112.758351,  48.656106)
NE = (-112.758351,  48.660910)

#Rumney Ranch RR3
NW = (-113.121304,  48.936337)
SW = (-113.121304,  48.921931)
SE = (-113.110147,  48.921931)
NE = (-113.110147,  48.936337)

#Rumney Ranch RR2
NW = (-113.132491,  48.932230)
SW = (-113.132491,  48.921931)
SE = (-113.121137,  48.921931)
NE = (-113.121137,  48.932230)

#Rumney Ranch RR
NW = (-113.141708,  48.928189)
SW = (-113.141708,  48.921931)
SE = (-113.131602,  48.921931)
NE = (-113.131602, 48.928189)

#Rumney Ranch RR
NW = (-113.141708,  48.928189)
SW = (-113.141708,  48.921931)
SE = (-113.131602,  48.921931)
NE = (-113.131602, 48.928189)

#Bernheim National Forest
NW = (-85.607730, 37.872445)
SW = (-85.607730, 37.869614)
SE = (-85.602016, 37.869614)
NE = (-85.602016, 37.872445)


#Pine Creek
NW = (-95.158374, 34.192683)
SW = (-95.158374, 34.190804)
SE = (-95.156150, 34.190804)
NE = (-95.156150, 34.192683)

#Acadia National Park
NW = (-68.273583, 44.366088)
SW = (-68.273583, 44.365399)  #fixed
SE = (-68.272327, 44.365399)
NE = (-68.272327, 44.366088)  #fixed

#Rumney Marsh
NW = (-71.0033967, 42.4345814)
SW = (-71.0034071, 42.4328122)
SE = (-71.0010287, 42.4328093)
NE = (-71.0009986, 42.4345665)

#RioBlanco/Buford
NW = (-107.646968,  39.988000)
SW = (-107.646968,  39.984158)
SE = (-107.642300, 39.984158)
NE = (-107.642300, 39.988000)

#Beaver Island
NW = (-85.531149,   45.644655)
SW = (-85.531232,  45.642625)  #fixed
SE = (-85.528579,   45.642469)
NE = (-85.527625,  45.644611)  #fixed

#Rumney Marsh ("EPSG:4326", "EPSG:26986")
NW = (-71.0033967, 42.4345814)
SW = (-71.0034071, 42.4328122)
SE = (-71.0010287, 42.4328093)
NE = (-71.0009986, 42.4345665)

#Skagit1
NE = (-122.357719,  48.302552)
SW = (-122.363813,  48.297440)
SE = (-122.356335,  48.297511)
NW = (-122.365551,  48.302530)


#Skagit2
NE = (-122.373676,  48.296786)
SW = (-122.379450,  48.292962)
SE = (-122.373585,  48.292989)
NW = (-122.379412,  48.296806)


#Minnesota 1
NW = (-90.050850, 47.992967)
SW = (-90.050884, 47.991889)  #fixed
SE = (-90.048032,  47.991838)
NE = (-90.048021,  47.992884)  #fixed

#Isle Royal
NE = (-88.901325,  47.958142)
SW = (-88.916436, 47.955043)
SE = (-88.914804, 47.953055)
NW = (-88.904798, 47.960272)


#Isle Royal
NW = (-88.904114, 47.960448)
SW = (-88.916436, 47.955043)  #fixed
SE = (-88.914804, 47.953055)
NE = (-88.904798, 47.960272)  #fixed

#Acadia National Park 2
NW = (-68.273123,  44.363276)
SW = (-68.273123,  44.361463)  #fixed
SE = (-68.269422,  44.361463)
NE = (-68.269422,  44.363276)  #fixed


"""




# Extract meta data

# Convert tif DEM into a CSV file

In [ ]:
#====================================================
# Convert tif image to csv file
#======================================================

RUN_EXPORT = True #set to True to execute code cell or False to skip

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer

if RUN_EXPORT:


    out_csv_name = file_name
    out_csv  = os.path.join(path, out_csv_name)  # or ".csv.gz" (see note below)
    
    # If you want gzip-compressed output, use:
    # out_csv = r"C:\path\to\dem_lonlat_elev.csv.gz"
    # and pass compression="gzip" in to_csv below.
    
    with rasterio.open(tif_path) as src:
        band = 1
        nodata = src.nodata
        crs_src = src.crs
        transform = src.transform
    
        if crs_src is None:
            raise RuntimeError("GeoTIFF has no CRS. You must define it before transforming to lon/lat.")
    
        # Transformer: raster CRS -> lon/lat
        to_ll = Transformer.from_crs(crs_src, "EPSG:4326", always_xy=True)
    
        first = True
    
        # Iterate in native internal blocks (fast + memory safe)
        for _, window in src.block_windows(band):
            z = src.read(band, window=window)
            if z.size == 0:
                continue

            # Build row/col indices for this window
            rows = np.arange(window.row_off, window.row_off + window.height)
            cols = np.arange(window.col_off, window.col_off + window.width)
            cc, rr = np.meshgrid(cols, rows)
    
            # Convert row/col -> raster CRS x/y using affine transform
            # Pixel centers: add 0.5 to col/row
            x = transform.c + (cc + 0.5) * transform.a + (rr + 0.5) * transform.b
            y = transform.f + (cc + 0.5) * transform.d + (rr + 0.5) * transform.e
    
            # Flatten
            x = x.ravel()
            y = y.ravel()
            zf = z.ravel()
    
            # Drop nodata / NaNs
            valid = np.isfinite(zf)
            if nodata is not None:
                valid &= (zf != nodata)
            if not np.any(valid):
                continue
    
            x = x[valid]
            y = y[valid]
            zf = zf[valid]
    
            # Transform to lon/lat
            lon, lat = to_ll.transform(x, y)
            lon, lat = to_ll.transform(x, y)

            df_out = pd.DataFrame({
                "x_m": x,          # raster CRS units (often meters)
                "y_m": y,
                "lon": lon,        # optional
                "lat": lat,        # optional
                "elevation": zf
            })
    

            df_out.to_csv(
                out_csv,
                mode="w" if first else "a",
                header=first,
                index=False,
                # compression="gzip"  # uncomment if out_csv ends with .gz
            )
            first = False
    
    print("Done. Wrote:", out_csv)

else:
    print("Skipping export block (RUN_EXPORT=False)")

In [ ]:
with rasterio.open(tif_path) as src:
    print("CRS:", src.crs)
    print("Resolution (x,y):", src.res)      # should be (~1.0, ~1.0) in meters if projected
    print("Transform:", src.transform)

In [ ]:
# -------------------------------
# Delimiter sniff
# -------------------------------
with open(csv_path, "r", newline="") as f:
    sample_text = "".join([next(f) for _ in range(20)])
sep = csv.Sniffer().sniff(sample_text).delimiter
print(f"Detected delimiter for {file_name!r}: {repr(sep)}")

In [ ]:
# -------------------------------
# Header detection (robust)
# -------------------------------
head = pd.read_csv(csv_path, sep=sep, header=None, nrows=1)
first_row = head.iloc[0].astype(str).str.lower().tolist()
has_header = any(s in ("x", "y", "elevation", "z") for s in first_row)

read_kwargs = dict(sep=sep)
if has_header:
    read_kwargs.update(dict(header=0))
else:
    read_kwargs.update(dict(header=None, names=["x", "y", "elevation"]))

In [ ]:
DATA_EPSG

In [ ]:
# ============================================================
# Elevation heatmap (auto-detect + auto-align CRS)
#   - Reads large CSVs in chunks
#   - Auto-detects coordinate columns (meters or lon/lat)
#   - Ensures plotting is done in projected meters (preferred for DEM work)
#   - Optional: overlays corners (lon/lat or projected)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Optional: only needed if we must convert lon/lat -> projected meters
from pyproj import Transformer

# -------------------------------
# USER SETTINGS (safe defaults)
# -------------------------------
chunksize = 1_000_000
max_cells = 10_000_000
bin_size_m_default = 1.0

CORNERS_ARE_LONLAT = True

if "DATA_CRS" not in globals() or DATA_CRS is None:
    with rasterio.open(tif_path) as src:
        DATA_CRS = src.crs
        DATA_EPSG = DATA_CRS.to_epsg() if DATA_CRS is not None else None
        CRS_LABEL = f"EPSG:{DATA_EPSG}" if DATA_EPSG is not None else str(DATA_CRS)

# -------------------------------
# Helper: detect columns
# -------------------------------
def pick_existing(cols, candidates):
    lower_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

probe = pd.read_csv(csv_path, nrows=50, low_memory=False)
cols = list(probe.columns)

elev_col = pick_existing(cols, ["elevation", "elev", "z", "dem", "height"])
if elev_col is None:
    elev_col = cols[2] if len(cols) >= 3 else None
if elev_col is None:
    raise RuntimeError("Could not detect elevation column. Add a column named 'elevation' (recommended).")

x_col = pick_existing(cols, ["x", "x_m", "easting"])
y_col = pick_existing(cols, ["y", "y_m", "northing"])

lon_col = pick_existing(cols, ["lon", "longitude"])
lat_col = pick_existing(cols, ["lat", "latitude"])

if x_col and y_col:
    coord_mode = "projected_xy"
elif lon_col and lat_col:
    coord_mode = "lonlat"
else:
    x_col = cols[0]
    y_col = cols[1]
    x_vals = pd.to_numeric(probe[x_col], errors="coerce")
    y_vals = pd.to_numeric(probe[y_col], errors="coerce")
    if (x_vals.abs().max() <= 180) and (y_vals.abs().max() <= 90):
        coord_mode = "lonlat_fallback"
        lon_col, lat_col = x_col, y_col
    else:
        coord_mode = "projected_xy_fallback"

print("=== CSV column detection ===")
print("  elevation:", elev_col)
print("  coord_mode:", coord_mode)
print("  x/y:", x_col, y_col)
print("  lon/lat:", lon_col, lat_col)
print("  target CRS:", CRS_LABEL)

ll_to_xy = None
if coord_mode in ("lonlat", "lonlat_fallback"):
    ll_to_xy = Transformer.from_crs("EPSG:4326", DATA_CRS, always_xy=True)

if coord_mode in ("projected_xy", "projected_xy_fallback"):
    usecols = [x_col, y_col, elev_col]
else:
    usecols = [lon_col, lat_col, elev_col]

xmin, xmax = np.inf, -np.inf
ymin, ymax = np.inf, -np.inf

for chunk in pd.read_csv(csv_path, chunksize=chunksize, usecols=usecols, low_memory=False):
    a = chunk.copy()

    if coord_mode in ("projected_xy", "projected_xy_fallback"):
        a["x_m"] = pd.to_numeric(a[x_col], errors="coerce")
        a["y_m"] = pd.to_numeric(a[y_col], errors="coerce")
    else:
        lon = pd.to_numeric(a[lon_col], errors="coerce").to_numpy()
        lat = pd.to_numeric(a[lat_col], errors="coerce").to_numpy()
        good = np.isfinite(lon) & np.isfinite(lat)
        if not np.any(good):
            continue
        x_m, y_m = ll_to_xy.transform(lon[good], lat[good])
        a = a.loc[good].copy()
        a["x_m"] = x_m
        a["y_m"] = y_m

    a["elev"] = pd.to_numeric(a[elev_col], errors="coerce")
    a = a.dropna(subset=["x_m", "y_m", "elev"])
    if a.empty:
        continue

    xmin = min(xmin, float(a["x_m"].min()))
    xmax = max(xmax, float(a["x_m"].max()))
    ymin = min(ymin, float(a["y_m"].min()))
    ymax = max(ymax, float(a["y_m"].max()))

if not np.isfinite(xmin) or not np.isfinite(ymin):
    raise RuntimeError("Could not determine coordinate extent from CSV (all NaN after parsing).")

print("\n=== Extent used for plotting (meters) ===")
print(f"  x_m: {xmin:.3f} -> {xmax:.3f}")
print(f"  y_m: {ymin:.3f} -> {ymax:.3f}")

bin_size_m = float(bin_size_m_default)

def grid_dims(xmin, xmax, ymin, ymax, bs):
    ncols = int(np.ceil((xmax - xmin) / bs)) + 1
    nrows = int(np.ceil((ymax - ymin) / bs)) + 1
    return nrows, ncols

nrows, ncols = grid_dims(xmin, xmax, ymin, ymax, bin_size_m)
cells = nrows * ncols

if cells > max_cells:
    scale = np.sqrt(cells / max_cells)
    bin_size_m *= scale
    nrows, ncols = grid_dims(xmin, xmax, ymin, ymax, bin_size_m)
    cells = nrows * ncols

print(f"\nBinning heatmap at ~{bin_size_m:.3f} m")
print(f"Grid: {ncols} x {nrows} = {cells:,} cells")

sum_z = np.zeros(cells, dtype=np.float64)
cnt = np.zeros(cells, dtype=np.uint32)

for chunk in pd.read_csv(csv_path, chunksize=chunksize, usecols=usecols, low_memory=False):
    a = chunk.copy()

    if coord_mode in ("projected_xy", "projected_xy_fallback"):
        x_m = pd.to_numeric(a[x_col], errors="coerce").to_numpy()
        y_m = pd.to_numeric(a[y_col], errors="coerce").to_numpy()
        good = np.isfinite(x_m) & np.isfinite(y_m)
        if not np.any(good):
            continue
        x_m = x_m[good]
        y_m = y_m[good]
        z = pd.to_numeric(a.loc[good, elev_col], errors="coerce").to_numpy()
    else:
        lon = pd.to_numeric(a[lon_col], errors="coerce").to_numpy()
        lat = pd.to_numeric(a[lat_col], errors="coerce").to_numpy()
        good = np.isfinite(lon) & np.isfinite(lat)
        if not np.any(good):
            continue
        x_m, y_m = ll_to_xy.transform(lon[good], lat[good])
        z = pd.to_numeric(a.loc[good, elev_col], errors="coerce").to_numpy()

    good2 = np.isfinite(z)
    if not np.any(good2):
        continue
    x_m = x_m[good2]
    y_m = y_m[good2]
    z = z[good2]

    ix = ((x_m - xmin) / bin_size_m).astype(np.int64)
    iy = ((y_m - ymin) / bin_size_m).astype(np.int64)

    valid = (ix >= 0) & (ix < ncols) & (iy >= 0) & (iy < nrows)
    if not np.any(valid):
        continue

    idx = iy[valid] * ncols + ix[valid]
    np.add.at(sum_z, idx, z[valid])
    np.add.at(cnt, idx, 1)

mean_z = np.full(cells, np.nan, dtype=np.float32)
mask = cnt > 0
mean_z[mask] = (sum_z[mask] / cnt[mask]).astype(np.float32)
Z = mean_z.reshape((nrows, ncols))

xmax_plot = xmin + bin_size_m * (ncols - 1)
ymax_plot = ymin + bin_size_m * (nrows - 1)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(
    Z,
    origin="lower",
    extent=[xmin, xmax_plot, ymin, ymax_plot],
    aspect="equal",
    cmap="terrain",
)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Elevation (m)")

ax.set_xlabel(f"X (m) [{CRS_LABEL}]")
ax.set_ylabel(f"Y (m) [{CRS_LABEL}]")
ax.set_title(f"{Path(csv_path).name} - Elevation Heatmap | source={data_source}")

if all(name in globals() for name in ["NW", "NE", "SW", "SE"]):
    if CORNERS_ARE_LONLAT:
        tf = Transformer.from_crs("EPSG:4326", DATA_CRS, always_xy=True)
        NW_xy = tf.transform(*NW)
        NE_xy = tf.transform(*NE)
        SW_xy = tf.transform(*SW)
        SE_xy = tf.transform(*SE)
    else:
        NW_xy, NE_xy, SW_xy, SE_xy = NW, NE, SW, SE

    for (pt, label) in [(NW_xy, "NW"), (NE_xy, "NE"), (SW_xy, "SW"), (SE_xy, "SE")]:
        ax.plot(pt[0], pt[1], marker="*", markersize=14)
        ax.text(pt[0], pt[1], label, fontsize=10, weight="bold",
                ha="left", va="bottom",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.8))
else:
    print("\n(no corners overlay - define NW/NE/SW/SE to plot them)")

plt.tight_layout()
plt.show()

out_png = os.path.join(os.path.dirname(csv_path), f"{Path(csv_path).stem}_heatmap.png")
fig.savefig(out_png, dpi=300, bbox_inches="tight")
print("Saved:", out_png)


In [ ]:
DATA_EPSG

In [ ]:
# ============================================================
# Probe + column detection
#   - Works whether CSV has headers or not
#   - Works whether CSV has extra columns or not
#   - Sets: XCOL, YCOL, ZCOL, COORD_MODE
# ============================================================

import numpy as np
import pandas as pd

# ---- Read a probe safely ----
probe = pd.read_csv(csv_path, nrows=10000, low_memory=False, **read_kwargs)

print("\n=== PROBE: raw columns ===")
print(list(probe.columns))

# ---- Helpers ----
def _pick_col(cols, candidates):
    """Return first matching column name from candidates (case-insensitive)."""
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

cols = list(probe.columns)

# Prefer explicit names if present
ZCOL = _pick_col(cols, ["elevation", "elev", "z", "dem", "height"])
XCOL = _pick_col(cols, ["x", "x_m", "easting"])
YCOL = _pick_col(cols, ["y", "y_m", "northing"])

LONCOL = _pick_col(cols, ["lon", "longitude"])
LATCOL = _pick_col(cols, ["lat", "latitude"])

# Fallbacks if the file is unnamed / old-style 3-col export
if ZCOL is None and len(cols) >= 3:
    ZCOL = cols[2]
if XCOL is None and YCOL is None and len(cols) >= 2:
    # if we have lon/lat named, use them
    if LONCOL and LATCOL:
        XCOL, YCOL = LONCOL, LATCOL
    else:
        # last resort: assume first two columns are coordinates
        XCOL, YCOL = cols[0], cols[1]

# ---- Coerce numeric for probe ranges ----
p = probe[[XCOL, YCOL, ZCOL]].copy()
p[XCOL] = pd.to_numeric(p[XCOL], errors="coerce")
p[YCOL] = pd.to_numeric(p[YCOL], errors="coerce")
p[ZCOL] = pd.to_numeric(p[ZCOL], errors="coerce")
p = p.dropna(subset=[XCOL, YCOL, ZCOL])

if p.empty:
    raise RuntimeError(
        "All x/y/elevation are NaN after parsing. "
        "Check delimiter/header (read_kwargs), and confirm column names."
    )

x_min_probe, x_max_probe = float(p[XCOL].min()), float(p[XCOL].max())
y_min_probe, y_max_probe = float(p[YCOL].min()), float(p[YCOL].max())
z_min_probe, z_max_probe = float(p[ZCOL].min()), float(p[ZCOL].max())

# ---- Guess coordinate mode ----
# If coords fall into lon/lat ranges, treat as degrees
if (abs(x_min_probe) <= 180 and abs(x_max_probe) <= 180 and
    abs(y_min_probe) <= 90  and abs(y_max_probe) <= 90):
    COORD_MODE = "lonlat_degrees"
else:
    COORD_MODE = "projected_meters"

print("\n=== PROBE (numeric) ===")
print(f"Using columns: XCOL='{XCOL}', YCOL='{YCOL}', ZCOL='{ZCOL}'")
print(f"COORD_MODE: {COORD_MODE}")
print(f"x: {x_min_probe} -> {x_max_probe}")
print(f"y: {y_min_probe} -> {y_max_probe}")
print(f"elevation: {z_min_probe} -> {z_max_probe}")

# ---- Sparse-grid warning (important for pivot-based slope) ----
rows = len(p)
ux = p[XCOL].nunique()
uy = p[YCOL].nunique()
grid_cells = ux * uy
fill_ratio = rows / grid_cells if grid_cells else np.nan

print("\n=== GRID DIAGNOSTICS ===")
print(f"rows (probe): {rows:,}")
print(f"unique x: {ux:,}")
print(f"unique y: {uy:,}")
print(f"unique_x * unique_y: {grid_cells:,}")
print(f"fill ratio (rows / grid_cells): {fill_ratio:.6e}")

if np.isfinite(fill_ratio) and fill_ratio < 0.01:
    print("  WARNING: This looks like a SPARSE point set, not a full raster grid.")
    print("    Pivoting to a 2D grid (pivot_table/unstack) will explode in size.")
    print("    For slope, compute from the GeoTIFF raster and sample at points instead.")

print("=====================================\n")


In [ ]:
# ============================================================
# STEP 1: Transform bbox lon/lat -> CSV CRS
# ============================================================
if "DATA_CRS" not in globals() or DATA_CRS is None:
    with rasterio.open(tif_path) as src:
        DATA_CRS = src.crs
        DATA_EPSG = DATA_CRS.to_epsg() if DATA_CRS is not None else None
        CRS_LABEL = f"EPSG:{DATA_EPSG}" if DATA_EPSG is not None else str(DATA_CRS)

tf_ll_to_data = Transformer.from_crs("EPSG:4326", DATA_CRS, always_xy=True)

ll_corners = np.array([NW, SW, SE, NE], dtype=float)
lon_c = ll_corners[:, 0]
lat_c = ll_corners[:, 1]
x_c, y_c = tf_ll_to_data.transform(lon_c, lat_c)

x_min_box, x_max_box = float(np.min(x_c)), float(np.max(x_c))
y_min_box, y_max_box = float(np.min(y_c)), float(np.max(y_c))

print("BBOX in CSV CRS:", x_min_box, x_max_box, y_min_box, y_max_box)

if abs(x_min_box) < 1000 and abs(x_max_box) < 1000 and DATA_EPSG is not None:
    raise RuntimeError(
        "Your bbox is still in lon/lat degrees. "
        "Set DATA_CRS/DATA_EPSG to the CRS of the CSV x/y."
    )

print(f"=== STUDY AREA BBOX in CSV CRS ({CRS_LABEL}) ===")
print(f"x_min: {x_min_box} | x_max: {x_max_box}")
print(f"y_min: {y_min_box} | y_max: {y_max_box}")
print("===============================================\n")


In [ ]:
probe_preview = pd.read_csv(csv_path, nrows=1000, low_memory=False, **read_kwargs)

probe_x_col = globals().get("XCOL", "x")
probe_y_col = globals().get("YCOL", "y")

if probe_x_col not in probe_preview.columns and len(probe_preview.columns) >= 2:
    probe_x_col = probe_preview.columns[0]
if probe_y_col not in probe_preview.columns and len(probe_preview.columns) >= 2:
    probe_y_col = probe_preview.columns[1]

probe_x = pd.to_numeric(probe_preview[probe_x_col], errors="coerce")
probe_y = pd.to_numeric(probe_preview[probe_y_col], errors="coerce")

x_min_probe = float(probe_x.min())
x_max_probe = float(probe_x.max())
y_min_probe = float(probe_y.min())
y_max_probe = float(probe_y.max())

print("BBOX (UTM):", x_min_box, x_max_box, y_min_box, y_max_box)
print("CSV probe x:", x_min_probe, x_max_probe, " | probe y:", y_min_probe, y_max_probe)
print("Note: probe y can be constant because it's the first scanline; that's normal.")


In [ ]:
DATA_EPSG

In [ ]:
# ============================================================
# STEP 1+2 (DROP-IN): Build bbox in CSV coordinates + stream-crop
#   - Handles CSVs with extra columns
#   - Handles lon/lat OR projected x/y
#   - Writes cropped CSV with columns: x, y, elevation
# ============================================================

import os
import numpy as np
import pandas as pd
from pyproj import Transformer

if "DATA_CRS" not in globals() or DATA_CRS is None:
    with rasterio.open(tif_path) as src:
        DATA_CRS = src.crs
        DATA_EPSG = DATA_CRS.to_epsg() if DATA_CRS is not None else None
        CRS_LABEL = f"EPSG:{DATA_EPSG}" if DATA_EPSG is not None else str(DATA_CRS)

probe = pd.read_csv(csv_path, nrows=1000, low_memory=False, **read_kwargs)
cols = list(probe.columns)

def pick_col(cols, candidates):
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

ZCOL = pick_col(cols, ["elevation", "elev", "z", "dem", "height"])
XCOL = pick_col(cols, ["x", "x_m", "easting"])
YCOL = pick_col(cols, ["y", "y_m", "northing"])
LONCOL = pick_col(cols, ["lon", "longitude"])
LATCOL = pick_col(cols, ["lat", "latitude"])

if ZCOL is None and len(cols) >= 3:
    ZCOL = cols[2]
if (XCOL is None or YCOL is None) and LONCOL and LATCOL:
    XCOL, YCOL = LONCOL, LATCOL
if XCOL is None and YCOL is None and len(cols) >= 2:
    XCOL, YCOL = cols[0], cols[1]

p = probe[[XCOL, YCOL, ZCOL]].copy()
p[XCOL] = pd.to_numeric(p[XCOL], errors="coerce")
p[YCOL] = pd.to_numeric(p[YCOL], errors="coerce")
p[ZCOL] = pd.to_numeric(p[ZCOL], errors="coerce")
p = p.dropna(subset=[XCOL, YCOL, ZCOL])

if p.empty:
    raise RuntimeError("Probe parse failed: all x/y/elevation are NaN. Check read_kwargs delimiter/header.")

x_min_probe, x_max_probe = float(p[XCOL].min()), float(p[XCOL].max())
y_min_probe, y_max_probe = float(p[YCOL].min()), float(p[YCOL].max())

if (abs(x_min_probe) <= 180 and abs(x_max_probe) <= 180 and
    abs(y_min_probe) <= 90 and abs(y_max_probe) <= 90):
    COORD_MODE = "lonlat_degrees"
else:
    COORD_MODE = "projected_meters"

print("\n=== COLUMN + COORD CHECK ===")
print(f"Using XCOL='{XCOL}', YCOL='{YCOL}', ZCOL='{ZCOL}'")
print(f"COORD_MODE: {COORD_MODE}")
print(f"CSV probe x: {x_min_probe} -> {x_max_probe}")
print(f"CSV probe y: {y_min_probe} -> {y_max_probe}")
print(f"Target CRS: {CRS_LABEL}")
print("================================\n")

ll_corners = np.array([NW, SW, SE, NE], dtype=float)
lon_c = ll_corners[:, 0]
lat_c = ll_corners[:, 1]

if COORD_MODE == "lonlat_degrees":
    x_min_box, x_max_box = float(np.min(lon_c)), float(np.max(lon_c))
    y_min_box, y_max_box = float(np.min(lat_c)), float(np.max(lat_c))
    print("BBOX will be used in lon/lat degrees (CSV is lon/lat).")
else:
    tf_ll_to_data = Transformer.from_crs("EPSG:4326", DATA_CRS, always_xy=True)
    x_c, y_c = tf_ll_to_data.transform(lon_c, lat_c)
    x_min_box, x_max_box = float(np.min(x_c)), float(np.max(x_c))
    y_min_box, y_max_box = float(np.min(y_c)), float(np.max(y_c))
    print(f"BBOX transformed to projected CRS {CRS_LABEL} (CSV is projected).")

print("\n=== BBOX IN CSV COORDS ===")
print(f"x_min_box: {x_min_box} | x_max_box: {x_max_box}")
print(f"y_min_box: {y_min_box} | y_max_box: {y_max_box}")
print("==========================\n")

base = os.path.splitext(file_name)[0]
cropped_path = os.path.join(path, f"{base}_cropped.csv")

if os.path.exists(cropped_path):
    os.remove(cropped_path)
    print("Removed existing cropped file:", cropped_path)

chunksize = 1_000_000
first = True
total_rows = 0
total_kept = 0
NODATA = -999999

for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunksize, low_memory=False, **read_kwargs), start=1):
    total_rows += len(chunk)
    chunk = chunk[[XCOL, YCOL, ZCOL]].copy()
    chunk.columns = ["x", "y", "elevation"]

    chunk["x"] = pd.to_numeric(chunk["x"], errors="coerce")
    chunk["y"] = pd.to_numeric(chunk["y"], errors="coerce")
    chunk["elevation"] = pd.to_numeric(chunk["elevation"], errors="coerce")
    chunk = chunk.dropna(subset=["x", "y", "elevation"])

    if NODATA is not None:
        chunk = chunk[chunk["elevation"] != NODATA]

    mask = (
        (chunk["x"] >= x_min_box) & (chunk["x"] <= x_max_box) &
        (chunk["y"] >= y_min_box) & (chunk["y"] <= y_max_box)
    )
    cropped = chunk.loc[mask]

    print(f"Chunk {i}: read {len(chunk):,} | kept {len(cropped):,}")

    if not cropped.empty:
        cropped.to_csv(cropped_path, mode="a", index=False, header=first)
        first = False
        total_kept += len(cropped)

print("\n=== STREAMING COMPLETE ===")
print(f"Total rows read : {total_rows:,}")
print(f"Total rows kept : {total_kept:,}")
print("Cropped file at :", cropped_path)

if total_kept == 0:
    raise RuntimeError(
        "No rows fell inside bbox.\n"
        f"- BBOX x: {x_min_box}..{x_max_box} | y: {y_min_box}..{y_max_box}\n"
        f"- PROBE x: {x_min_probe}..{x_max_probe} | y: {y_min_probe}..{y_max_probe}\n"
        "Likely causes:\n"
        "  (1) bbox corners are wrong, or\n"
        "  (2) DATA_CRS is wrong, or\n"
        "  (3) CSV coords are lon/lat but you assumed projected (or vice versa).\n"
    )


In [ ]:
# ============================================================
# STEP 3 (DROP-IN): Quick plot of cropped data (1m binning)
#   - Robust to CSV being lon/lat OR projected meters
#   - Always plots in projected CRS so 1m stays 1m
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer
from pathlib import Path

if "DATA_CRS" not in globals() or DATA_CRS is None:
    with rasterio.open(tif_path) as src:
        DATA_CRS = src.crs
        DATA_EPSG = DATA_CRS.to_epsg() if DATA_CRS is not None else None
        CRS_LABEL = f"EPSG:{DATA_EPSG}" if DATA_EPSG is not None else str(DATA_CRS)

design_area = pd.read_csv(cropped_path, low_memory=False)

need = ["x", "y", "elevation"]
missing = [c for c in need if c not in design_area.columns]
if missing:
    raise RuntimeError(f"cropped CSV must contain columns {need}. Missing: {missing}")

design_area = design_area[need].copy()
for c in need:
    design_area[c] = pd.to_numeric(design_area[c], errors="coerce")
design_area = design_area.dropna(subset=need)

print(f"Cropped data: {len(design_area):,} rows")
print(f"X range: {design_area['x'].min():.6f} to {design_area['x'].max():.6f}")
print(f"Y range: {design_area['y'].min():.6f} to {design_area['y'].max():.6f}")

xmn, xmx = float(design_area["x"].min()), float(design_area["x"].max())
ymn, ymx = float(design_area["y"].min()), float(design_area["y"].max())

is_lonlat = (abs(xmn) <= 180 and abs(xmx) <= 180 and abs(ymn) <= 90 and abs(ymx) <= 90)

if is_lonlat:
    print("Detected CSV coordinates: lon/lat degrees -> projecting to meters for plotting/binning.")
    tf_to_m = Transformer.from_crs("EPSG:4326", DATA_CRS, always_xy=True)
    xm, ym = tf_to_m.transform(design_area["x"].to_numpy(), design_area["y"].to_numpy())
    X = np.asarray(xm, dtype=np.float64)
    Y = np.asarray(ym, dtype=np.float64)
else:
    print(f"Detected CSV coordinates: projected units (assumed {CRS_LABEL}).")
    X = design_area["x"].to_numpy(np.float64)
    Y = design_area["y"].to_numpy(np.float64)

Zvals = design_area["elevation"].to_numpy(np.float64)

bin_size_m = 1.0
max_cells = 10_000_000

xmin, xmax = float(np.min(X)), float(np.max(X))
ymin, ymax = float(np.min(Y)), float(np.max(Y))

def grid_dims(xmin, xmax, ymin, ymax, bs):
    ncols = int(np.ceil((xmax - xmin) / bs)) + 1
    nrows = int(np.ceil((ymax - ymin) / bs)) + 1
    return nrows, ncols

nrows, ncols = grid_dims(xmin, xmax, ymin, ymax, bin_size_m)
cells = nrows * ncols

if cells > max_cells:
    scale = np.sqrt(cells / max_cells)
    bin_size_m *= scale
    nrows, ncols = grid_dims(xmin, xmax, ymin, ymax, bin_size_m)
    cells = nrows * ncols

print(f"\nBinning at {bin_size_m:.3f} m")
print(f"Grid: {ncols} x {nrows} = {cells:,} cells\n")

sum_z = np.zeros(cells, dtype=np.float64)
cnt = np.zeros(cells, dtype=np.uint32)

ix = ((X - xmin) / bin_size_m).astype(np.int64)
iy = ((Y - ymin) / bin_size_m).astype(np.int64)

valid = (ix >= 0) & (ix < ncols) & (iy >= 0) & (iy < nrows)
idx = iy[valid] * ncols + ix[valid]

np.add.at(sum_z, idx, Zvals[valid])
np.add.at(cnt, idx, 1)

mean_z = np.full(cells, np.nan, dtype=np.float32)
mask = cnt > 0
mean_z[mask] = (sum_z[mask] / cnt[mask]).astype(np.float32)
Zgrid = mean_z.reshape((nrows, ncols))

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    Zgrid,
    origin="lower",
    extent=[xmin, xmin + bin_size_m * (ncols - 1), ymin, ymin + bin_size_m * (nrows - 1)],
    aspect="equal",
    cmap="terrain",
)
plt.colorbar(im, ax=ax, label="Elevation")
ax.set_title(f"Cropped elevation heatmap (projected): {Path(cropped_path).name} | {CRS_LABEL}")
ax.set_xlabel(f"X [{CRS_LABEL}]")
ax.set_ylabel(f"Y [{CRS_LABEL}]")
plt.tight_layout()
plt.show()


# Convert CSV file to ASC file

In [ ]:
csv_path = cropped_path
out_asc_path = os.path.join(path, f"{base}.asc")

In [ ]:
csv_path

In [ ]:


def _infer_cellsize(x, y):
    #Robust cellsize guess from median unique diffs; fallback to span/500.
    xu = np.unique(np.sort(x))
    yu = np.unique(np.sort(y))
    ests = []
    if len(xu) > 1: ests.append(np.median(np.diff(xu)))
    if len(yu) > 1: ests.append(np.median(np.diff(yu)))
    if ests:
        # Avoid weird huge steps if there are occasional large gaps
        return float(np.nanmin(ests))
    span = max(x.max()-x.min(), y.max()-y.min())
    return float(span / 500.0 if span > 0 else 1.0)

def _grid_axes(x, y, cellsize, buffer_cells=0):
    x_min, x_max = float(x.min()), float(x.max())
    y_min, y_max = float(y.min()), float(y.max())
    x_min -= buffer_cells * cellsize
    y_min -= buffer_cells * cellsize
    x_max += buffer_cells * cellsize
    y_max += buffer_cells * cellsize
    xg = np.arange(x_min, x_max + 0.5*cellsize, cellsize)
    yg = np.arange(y_min, y_max + 0.5*cellsize, cellsize)
    return xg, yg

def _interpolate_idw(x, y, z, Xg, Yg, k=12, power=2.0, nodata=np.nan):
    #Fast IDW via KD-tree (k nearest)
    pts = np.column_stack([x, y])
    tree = cKDTree(pts)
    q = np.column_stack([Xg.ravel(), Yg.ravel()])
    dists, idxs = tree.query(q, k=min(k, len(pts)))
    # Ensure 2D
    if dists.ndim == 1:
        dists = dists[:, None]
        idxs = idxs[:, None]
    weights = 1.0 / np.maximum(dists, 1e-12)**power
    z_neighbors = z[idxs]
    z_num = np.sum(weights * z_neighbors, axis=1)
    z_den = np.sum(weights, axis=1)
    Zi = z_num / np.maximum(z_den, 1e-12)
    Zi = Zi.reshape(Xg.shape)
    # If all neighbors are identical NaNs (shouldn't happen), set nodata
    Zi[~np.isfinite(Zi)] = nodata
    return Zi

def xyz_csv_to_asc_interpolated(csv_path, out_asc_path,
                                x_col="x", y_col="y", z_col="elevation",
                                cellsize=None,
                                method="nearest",   # 'nearest'|'linear'|'cubic'|'idw'
                                idw_k=12, idw_power=2.0,
                                buffer_cells=0,
                                nodata=-9999,
                                round_decimals=9):
    """
    #Create an interpolated ESRI ASCII grid from scattered XYZ points.
    """
    df = pd.read_csv(csv_path)
    # Clean/round to reduce jitter
    df = df[[x_col, y_col, z_col]].dropna()
    df[x_col] = df[x_col].round(round_decimals)
    df[y_col] = df[y_col].round(round_decimals)

    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    z = df[z_col].to_numpy(dtype=float)

    if cellsize is None:
        cellsize = _infer_cellsize(x, y)

    xg, yg = _grid_axes(x, y, cellsize, buffer_cells=buffer_cells)
    Xg, Yg = np.meshgrid(xg, yg)

    # Interpolate
    if method.lower() == "idw":
        Zg = _interpolate_idw(x, y, z, Xg, Yg, k=idw_k, power=idw_power, nodata=np.nan)
    else:
        Zg = griddata(points=np.column_stack([x, y]),
                      values=z,
                      xi=(Xg, Yg),
                      method=method.lower())

    # Replace NaNs (uninterpolated) with NODATA
    Zg = np.where(np.isnan(Zg), nodata, Zg)

    # ESRI ASCII expects rows written from maxY -> minY
    Z_top_to_bottom = np.flipud(Zg)

    nrows, ncols = Zg.shape
    xllcorner = float(xg.min())
    yllcorner = float(yg.min())

    with open(out_asc_path, "w") as f:
        f.write(f"ncols         {ncols}\n")
        f.write(f"nrows         {nrows}\n")
        f.write(f"xllcorner     {xllcorner}\n")
        f.write(f"yllcorner     {yllcorner}\n")
        f.write(f"cellsize      {cellsize}\n")
        f.write(f"NODATA_value  {nodata}\n")
        for i in range(nrows):
            row = Z_top_to_bottom[i, :]
            f.write(" ".join(str(float(v)) for v in row) + "\n")

    print(f" Wrote {out_asc_path}  "
          f"({ncols}x{nrows}, cellsize={cellsize}, method={method})")


xyz_csv_to_asc_interpolated(csv_path, out_asc_path)

# Hydromodeling

In [ ]:
def create_grid_FIXED(
    df: pd.DataFrame,
    grid_spacing: float = 1.0,
    x_col: str = "x",
    y_col: str = "y",
    value_cols=("elevation",),   # add more like ("elevation","width_m","depth_m")
    agg: str = "first",          # or "mean", "max", etc. if multiple points fall in one cell
    return_df_with_indices: bool = True
):
    """
    Build a rectilinear grid from scattered x,y points in `df`.
    - Snaps coordinates to the specified `grid_spacing`
    - Returns 2D arrays for each column in `value_cols`
    - Preserves Y increasing upward (origin='lower' for imshow)

    Returns:
        grid = {
            "x_coords": np.ndarray [nx],
            "y_coords": np.ndarray [ny],
            "X": np.ndarray [ny, nx],
            "Y": np.ndarray [ny, nx],
            "<col>": np.ndarray [ny, nx] for each col in value_cols,
            "mask": np.ndarray [ny, nx]  # True where at least one value exists
            "x_to_ix": dict, "y_to_iy": dict
        }
        (and optionally a copy of df with snapped coords + ix/iy)
    """
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    # 1) snap coordinates to grid
    def _snap(v, s):
        return np.round(np.asarray(v, dtype=float) / s) * s

    x_snap = _snap(df[x_col].values, grid_spacing)
    y_snap = _snap(df[y_col].values, grid_spacing)

    # 2) coordinate axes
    x_coords = np.unique(x_snap)
    y_coords = np.unique(y_snap)
    nx, ny = len(x_coords), len(y_coords)

    # 3) index maps
    x_to_ix = {x: i for i, x in enumerate(x_coords)}
    y_to_iy = {y: i for i, y in enumerate(y_coords)}

    # 4) pivot each requested value column onto the full grid
    #    Use full reindex so missing cells become NaN
    grid = {
        "x_coords": x_coords,
        "y_coords": y_coords,
        "x_to_ix": x_to_ix,
        "y_to_iy": y_to_iy,
    }

    # Small helper to create a uniform 2D mesh (useful for plotting)
    X, Y = np.meshgrid(x_coords, y_coords)
    grid["X"], grid["Y"] = X, Y

    # Temporary working frame with snapped coords
    work = df.copy()
    work["_x_snap"] = x_snap
    work["_y_snap"] = y_snap

    # If label-like columns exist that are non-numeric, you can still pivot them with agg='first'
    for col in value_cols:
        if col not in work.columns:
            # create a placeholder if missing
            grid[col] = np.full((ny, nx), np.nan, dtype=float)
            continue

        # Choose dtype-aware aggregation; pandas pivot_table will handle strings too with 'first'
        pv = (
            work.pivot_table(
                index="_y_snap",
                columns="_x_snap",
                values=col,
                aggfunc=agg,
                dropna=False,
            )
            .reindex(index=y_coords, columns=x_coords)
        )

        # Convert to array; for non-numeric (e.g., 'label') keep object dtype
        arr = pv.values
        grid[col] = arr

    # 5) presence mask: any non-NaN among numeric columns (fall back to all columns)
    cand_cols = [c for c in value_cols if np.issubdtype(np.asarray(grid[c]).dtype, np.number)]
    if not cand_cols:
        # if all are non-numeric, fall back to existence in the df
        presence = ~pd.isna(
            work.pivot_table(index="_y_snap", columns="_x_snap", values=value_cols[0], aggfunc="first")
             .reindex(index=y_coords, columns=x_coords)
             .values
        )
    else:
        stacks = [np.asarray(grid[c]) for c in cand_cols]
        presence = np.any(~np.isnan(stacks), axis=0) if len(stacks) > 1 else ~np.isnan(stacks[0])
    grid["mask"] = presence

    # 6) (optional) annotate df with snapped coords and grid indices
    if return_df_with_indices:
        work["ix"] = [x_to_ix[x] for x in work["_x_snap"]]
        work["iy"] = [y_to_iy[y] for y in work["_y_snap"]]
        return grid, work.drop(columns=["_x_snap", "_y_snap"])

    return grid

In [ ]:
from pyproj import Transformer
from landlab import RasterModelGrid
from landlab.components import FlowAccumulator

In [ ]:
# ============================================================================
# STEP 2+3 (DROP-IN): Build grid in METERS, run flow accumulation, plot
# ============================================================================

print("Loading cropped data...")
df = pd.read_csv(cropped_path, low_memory=False).copy()
df = df.iloc[:, :3].copy()
df.columns = ["x", "y", "elevation"]

for c in ["x", "y", "elevation"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["x", "y", "elevation"])
print(f"Loaded {len(df):,} points")

# ---- Detect lon/lat vs projected meters ----
xmn, xmx = float(df["x"].min()), float(df["x"].max())
ymn, ymx = float(df["y"].min()), float(df["y"].max())
is_lonlat = (abs(xmn) <= 180 and abs(xmx) <= 180 and abs(ymn) <= 90 and abs(ymx) <= 90)

if is_lonlat:
    target_crs = globals().get("DATA_CRS", globals().get("dem_crs"))
    target_label = globals().get("CRS_LABEL", str(target_crs))
    print(f"Detected lon/lat CSV -> projecting to {target_label}")
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    X, Y = tf.transform(df["x"].to_numpy(), df["y"].to_numpy())
    df["x"] = X
    df["y"] = Y
else:
    target_label = globals().get("CRS_LABEL", globals().get("DATA_EPSG", "projected CRS"))
    print(f"Detected projected CSV (assumed {target_label})")

# ---- Infer grid spacing in meters (should be ~1.0 for your GeoTIFF) ----
def infer_step(vals):
    u = np.unique(np.round(vals.astype(float), 6))
    u.sort()
    d = np.diff(u)
    d = d[d > 0]
    if d.size == 0:
        return np.nan
    r = np.round(d, 6)
    uniq, cnt = np.unique(r, return_counts=True)
    return float(uniq[np.argmax(cnt)])

dx_x = infer_step(df["x"].values)
dx_y = infer_step(df["y"].values)
native_grid_spacing_m = np.nanmin([dx_x, dx_y])
if not np.isfinite(native_grid_spacing_m) or native_grid_spacing_m <= 0:
    native_grid_spacing_m = 1.0

# ---- Optional hydro-only coarsening ----
HYDRO_COARSEN_FACTOR = globals().get("HYDRO_COARSEN_FACTOR", 1)
if HYDRO_COARSEN_FACTOR is None or HYDRO_COARSEN_FACTOR < 1:
    HYDRO_COARSEN_FACTOR = 1

grid_spacing_m = float(native_grid_spacing_m) * float(HYDRO_COARSEN_FACTOR)

print(
    f"Estimated native grid spacing: {native_grid_spacing_m:.3f} m "
    f"(x_step={dx_x}, y_step={dx_y})"
)
if HYDRO_COARSEN_FACTOR > 1:
    print(
        f"Hydro coarsening enabled: factor={HYDRO_COARSEN_FACTOR} "
        f"-> hydro grid spacing={grid_spacing_m:.3f} m"
    )
else:
    print(f"Hydro coarsening disabled: using {grid_spacing_m:.3f} m")

# ---- Build snapped grid (meters) ----
grid_dict, df_with_ixiy = create_grid_FIXED(
    df,
    grid_spacing=grid_spacing_m,
    value_cols=("elevation",),
    agg="mean",
)

x_coords = grid_dict["x_coords"]
y_coords = grid_dict["y_coords"]
nrows, ncols = len(y_coords), len(x_coords)
print(f"Grid: {nrows} rows x {ncols} cols = {nrows*ncols:,} nodes")

if nrows < 4 or ncols < 4:
    raise ValueError(f"Grid too small ({nrows}x{ncols}). Check cropping / spacing.")

# ---- Create Landlab grid in METERS ----
try:
    mg = RasterModelGrid((nrows, ncols), xy_spacing=grid_spacing_m)
except TypeError:
    mg = RasterModelGrid((nrows, ncols), grid_spacing_m)

# Make absolutely sure the DEM is a writable, C-contiguous float array
Z2d = np.array(grid_dict["elevation"], dtype=float, copy=True, order="C")
nan_mask = ~np.isfinite(Z2d)
print(f"NaNs in Z grid: {nan_mask.sum():,} / {Z2d.size:,}")

if nan_mask.any():
    mg.status_at_node[nan_mask.ravel(order="C")] = mg.BC_NODE_IS_CLOSED
    safe_min = np.nanmin(Z2d)
    Z2d[nan_mask] = safe_min

z_nodes = np.array(Z2d.ravel(order="C"), dtype=float, copy=True)

print("z_nodes writable:", z_nodes.flags.writeable)
print("z_nodes C contiguous:", z_nodes.flags.c_contiguous)

mg.add_field(
    "topographic__elevation",
    z_nodes,
    at="node",
    clobber=True,
    copy=True,
)

# Open boundaries (water can leave domain)
mg.set_closed_boundaries_at_grid_edges(False, False, False, False)

# ---- Run flow accumulation ----
print("Running FlowAccumulator...")
fa = FlowAccumulator(mg, flow_director="D8", depression_finder="DepressionFinderAndRouter")
fa.run_one_step()

drainage_area_m2 = mg.at_node["drainage_area"].copy()
print("Done.")
print(
    "Drainage area (m^2):",
    "min=", np.nanmin(drainage_area_m2),
    "max=", np.nanmax(drainage_area_m2),
    "mean=", np.nanmean(drainage_area_m2),
)

# ---- Plot (use robust scaling so you don't get all one color) ----
da2d = drainage_area_m2.reshape((nrows, ncols))
closed2d = (mg.status_at_node.reshape((nrows, ncols)) == mg.BC_NODE_IS_CLOSED)
da2d = np.where(closed2d, np.nan, da2d)

cell_area = grid_spacing_m * grid_spacing_m
log_da = np.log10(da2d + cell_area)

vmin, vmax = np.nanpercentile(log_da, [5, 99.5])

xmin, xmax = float(x_coords.min()), float(x_coords.max())
ymin, ymax = float(y_coords.min()), float(y_coords.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(
    Z2d, origin="lower",
    extent=[xmin, xmax, ymin, ymax],
    cmap="terrain", aspect="auto"
)
axes[0].set_title(f"DEM (Filled) - {nrows}x{ncols} (~{grid_spacing_m:.1f} m)")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")

im = axes[1].imshow(
    log_da, origin="lower",
    extent=[xmin, xmax, ymin, ymax],
    cmap="Blues", aspect="auto",
    vmin=vmin, vmax=vmax
)
axes[1].set_title(f"Flow Accumulation (log10 m^2) | max={np.nanmax(da2d)/1e6:.2f} km^2")
axes[1].set_xlabel("Easting (m)")
axes[1].set_ylabel("Northing (m)")
plt.colorbar(im, ax=axes[1], label="log10(drainage area m^2)")

plt.tight_layout()
plt.show()


In [ ]:


# ============================================================================
# STEP 4: Map drainage area back to CSV and save
# ============================================================================

print("\nMapping drainage area back to original points...")

# Reshape drainage_area back to 2D grid
da2d = drainage_area_m2.reshape((nrows, ncols))

# Use the df_with_ixiy dataframe (which has ix, iy grid indices)
# to look up the corresponding drainage area for each point
df_with_ixiy["drainage_area_m2"] = [
    da2d[iy, ix] if (0 <= iy < nrows and 0 <= ix < ncols) else np.nan
    for ix, iy in zip(df_with_ixiy["ix"], df_with_ixiy["iy"])
]

# Optional: also add drainage area in km for easier interpretation
df_with_ixiy["drainage_area_km2"] = df_with_ixiy["drainage_area_m2"] / 1e6

print(f"Added drainage area to {len(df_with_ixiy):,} points")
print(f"  Range: {df_with_ixiy['drainage_area_km2'].min():.6f} to {df_with_ixiy['drainage_area_km2'].max():.2f} km")


# If you want to update the original cropped file instead:
df_with_ixiy.to_csv(cropped_path, index=False)
print(f"\nUpdated: {cropped_path}")

In [ ]:
# ====================================
# Visualize Flow Accumulation (Drainage Area)
# ====================================
ext = ".csv"
png = ".png"
drainage_file = f"{base}_cropped{ext}"
heat_file = f"{base}_drainage_heatmap{png}"
csv_path = os.path.join(path, drainage_file)
plot_heatmap_path = os.path.join(path, heat_file)

print(f"\nLoading drainage area data from: {csv_path}")

df = pd.read_csv(csv_path, low_memory=False)

needed = ["x", "y", "drainage_area_m2"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing columns: {missing}\nFound columns: {list(df.columns)}")

df = df[["x", "y", "drainage_area_m2", "drainage_area_km2"]].copy()
df["x"] = pd.to_numeric(df["x"], errors="coerce")
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df["drainage_area_m2"] = pd.to_numeric(df["drainage_area_m2"], errors="coerce")
df["drainage_area_km2"] = pd.to_numeric(df["drainage_area_km2"], errors="coerce")
df = df.dropna(subset=["x", "y", "drainage_area_m2"])

print(f"Loaded: {len(df):,} rows")

xs = np.sort(df["x"].unique())
ys = np.sort(df["y"].unique())

if len(xs) < 2 or len(ys) < 2:
    raise ValueError("Not enough unique x/y values to form a grid.")

dx = float(np.median(np.diff(xs)))
dy = float(np.median(np.diff(ys)))
x0, y0 = xs[0], ys[0]
nx, ny = len(xs), len(ys)

print(f"Grid: {ny} rows x {nx} cols")
print(f"Spacing: dx={dx:.3f} m, dy={dy:.3f} m")

ix = np.rint((df["x"].to_numpy() - x0) / dx).astype(np.int64)
iy = np.rint((df["y"].to_numpy() - y0) / dy).astype(np.int64)
da = df["drainage_area_m2"].to_numpy(dtype=np.float64)

valid = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny) & np.isfinite(da)
ix, iy, da = ix[valid], iy[valid], da[valid]

sum_da = np.zeros((ny, nx), dtype=np.float64)
cnt_da = np.zeros((ny, nx), dtype=np.uint32)
np.add.at(sum_da, (iy, ix), da)
np.add.at(cnt_da, (iy, ix), 1)

DA = np.full((ny, nx), np.nan, dtype=np.float64)
mask = cnt_da > 0
DA[mask] = sum_da[mask] / cnt_da[mask]

print(f"Drainage area range: {np.nanmin(DA)/1e6:.6f} to {np.nanmax(DA)/1e6:.2f} km^2")

extent = [xs.min() - dx / 2, xs.max() + dx / 2, ys.min() - dy / 2, ys.max() + dy / 2]
cell_area = dx * dy
log_DA = np.log10(DA + cell_area)
vmin, vmax = np.nanpercentile(log_DA, [5, 99.5])

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(
    log_DA,
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap="Blues",
    vmin=vmin,
    vmax=vmax
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log(Drainage Area m^2)", fontsize=11)

ax.set_xlabel("Easting (m)", fontsize=11)
ax.set_ylabel("Northing (m)", fontsize=11)
ax.set_title(
    f"Flow Accumulation {base}\nmax = {np.nanmax(DA)/1e6:.2f} km^2",
    fontsize=12,
    fontweight="bold"
)

plt.tight_layout()
plt.savefig(plot_heatmap_path, dpi=200, bbox_inches="tight")
print(f"Saved heatmap to: {plot_heatmap_path}")
plt.show()


In [ ]:
# ============================================================================
# STEP 4: Extract Drainage
# ============================================================================

threshold_percentile = 0.1 # Adjust 0.1-99.9 as needed
threshold = np.percentile(drainage_area_m2, threshold_percentile)
stream_mask = drainage_area_m2 >= threshold

print(f"\nStream extraction:")
print(f"  Threshold: {threshold:.2f} m ({threshold_percentile}th percentile)")
print(f"  Stream cells: {stream_mask.sum():,}")

In [ ]:
# ============================================================================
# STEP 5: Visualize Storm event
# ============================================================================

# --- Use the Landlab grid object ---
# mg: RasterModelGrid with 'topographic__elevation' and FlowAccumulator already run
elevation = mg.at_node['topographic__elevation']

# If you don't already have a stream mask, make one from drainage area (top 2%)
if 'stream_mask' not in globals():
    da = mg.at_node['drainage_area']
    # use only finite/core nodes to compute a robust threshold
    valid = np.isfinite(da)
    thresh = np.quantile(da[valid], 0.98)
    stream_mask = da >= thresh

# Convenience
nrows, ncols = mg.shape
dx = mg.dx if hasattr(mg, 'dx') else 1.0

# imshow expects pixel edges; approximate from node coords
xmin, xmax = mg.x_of_node.min(), mg.x_of_node.max()
ymin, ymax = mg.y_of_node.min(), mg.y_of_node.max()
extent = [xmin - dx/2, xmax + dx/2, ymin - dx/2, ymax + dx/2]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: DEM + stream overlay ---
ax1 = axes[0]
im1 = ax1.imshow(
    elevation.reshape(nrows, ncols),
    origin='lower',
    cmap='terrain',
    extent=extent,
    alpha=0.6
)

stream_nodes = np.flatnonzero(stream_mask)
ax1.scatter(
    mg.x_of_node[stream_nodes],
    mg.y_of_node[stream_nodes],
    c='red', s=3, alpha=0.8, label='Stream'
)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Extracted Stream Network', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right')
plt.colorbar(im1, ax=ax1, label='Elevation (m)')

# --- Right: Stream-only elevation (masked elsewhere as NaN) ---
ax2 = axes[1]
stream_elev = np.full(mg.number_of_nodes, np.nan, dtype=float)
stream_elev[stream_mask] = elevation[stream_mask]
im2 = ax2.imshow(
    stream_elev.reshape(nrows, ncols),
    origin='lower',
    cmap='viridis',
    extent=extent
)
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('Stream Channel (isolated)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax2, label='Elevation (m)')

plt.tight_layout()
# Save plot as PNG in the same directory as the CSV
plot_filename = f"{base}_stream_channel_isolated.png"
plot_path = os.path.join(path, plot_filename)
plt.savefig(plot_path, dpi=200, bbox_inches='tight')
print(f"Saved plot to: {plot_path}")
plt.show()


In [ ]:
# ============================================================================
# STEP 4B+6: Hydrodynamic features  write back to design CSV (+ optional stream CSV)
# ============================================================================

# --- safety/default ---
DEFAULT_THRESHOLD_PERCENTILE = 0.1
try:
    threshold_percentile
except NameError:
    threshold_percentile = DEFAULT_THRESHOLD_PERCENTILE

# --- 1) Node-wise slope from elevation ---
nrows, ncols = mg.shape
dx = float(getattr(mg, "dx", 1.0))

elev_nodes = mg.at_node["topographic__elevation"]
Z2 = elev_nodes.reshape(nrows, ncols)

# gradient-based slope magnitude
dZdy = np.gradient(Z2, dx, axis=0)
dZdx = np.gradient(Z2, dx, axis=1)
slope_nodes = np.hypot(dZdx, dZdy).ravel()

# --- 2) Drainage-area percentile + stream mask ---
drainage_area = mg.at_node["drainage_area"]
closed = (mg.status_at_node == mg.BC_NODE_IS_CLOSED)
valid = (~closed) & np.isfinite(drainage_area)

sorted_da = np.sort(drainage_area[valid])

def _percentile_rank(values, ref_sorted):
    # empirical CDF, right-inclusive
    idx = np.searchsorted(ref_sorted, values, side="right")
    return 100.0 * (idx / ref_sorted.size)

percentile_all = np.full(drainage_area.shape, np.nan, dtype=float)
percentile_all[valid] = _percentile_rank(drainage_area[valid], sorted_da)

cutoff = np.percentile(drainage_area[valid], threshold_percentile)
stream_mask = drainage_area >= cutoff

# --- 3) Width/depth (only for stream cells; NaN elsewhere) ---
width_all = np.full_like(drainage_area, np.nan, dtype=float)
depth_all = np.full_like(drainage_area, np.nan, dtype=float)

da_km2 = drainage_area / 1e6
width_all[stream_mask] = 0.4 * np.sqrt(np.clip(da_km2[stream_mask], 0, None))
depth_all[stream_mask] = 0.3 * np.power(np.clip(width_all[stream_mask], 0, None), 0.4)

# --- 4) Map node arrays  rows in your input DataFrame ---
node_id = (
    df_with_ixiy["iy"].astype(int).to_numpy() * ncols
    + df_with_ixiy["ix"].astype(int).to_numpy()
)

df_out = df_with_ixiy.copy()
df_out["drainage_area"] = drainage_area[node_id]
df_out["slope"]         = slope_nodes[node_id]
df_out["width_m"]       = width_all[node_id]
df_out["depth_m"]       = depth_all[node_id]
df_out["streamline"]    = np.where(stream_mask[node_id], "yes", "no")
df_out["percentile"]    = np.round(percentile_all[node_id], 2)  # 0100

# --- 4b) Choose a REAL file path to save to (NOT the directory!) ---

# Use the cropped Chile DEM CSV as the base, and append "_hydro"
# Assumes `cropped_path` and `path` are defined from earlier cells.
base_cropped = os.path.splitext(os.path.basename(cropped_path))[0]
ext_cropped  = os.path.splitext(os.path.basename(cropped_path))[1]

out_path = os.path.join(path, f"{base_cropped}_hydro{ext_cropped}")

# drop helper indices and write CSV
df_out.drop(columns=["ix", "iy"], errors="ignore").to_csv(out_path, index=False)

print(" STEP 4B+6 complete.")
print(f"  Updated (hydro-annotated design CSV): {out_path}")
print(f"  Stream cutoff: {threshold_percentile}th percentile ")
print(f"  Stream cells:  {int(stream_mask.sum()):,} / {mg.number_of_nodes:,}")

# --- 5) (Optional) export streams-only CSV, like old Step 6 ---
try:
    # find an original design-area dataframe for offsets (if available)
    _design_df = None
    for cand in ("design_area", "df_design", "df"):
        if cand in globals():
            _design_df = globals()[cand]
            break

    x_offset = float(_design_df["x"].min()) if _design_df is not None else 0.0
    y_offset = float(_design_df["y"].min()) if _design_df is not None else 0.0

    stream_nodes = np.flatnonzero(stream_mask)
    stream_points = pd.DataFrame({
        "x": mg.x_of_node[stream_nodes] + x_offset,
        "y": mg.y_of_node[stream_nodes] + y_offset,
        "elevation":     elev_nodes[stream_nodes],
        "drainage_area": drainage_area[stream_nodes],
        "slope":         slope_nodes[stream_nodes],
        "width_m":       width_all[stream_nodes],
        "depth_m":       depth_all[stream_nodes],
        "percentile":    percentile_all[stream_nodes],
    })

    # Save a streams-only file next to the hydro CSV
    stream_csv = os.path.join(path, f"{base_cropped}_streams{ext_cropped}")
    stream_points.to_csv(stream_csv, index=False)
    print(f"  Also saved streams-only CSV: {stream_csv}")
except Exception as e:
    print(f"  (Optional streams-only export skipped: {e})")


In [ ]:
# ============================================================================
# STEP 7B: Align stream coords to design frame via box-to-box affine transform,
#          then nearest-neighbor join to update design_area
#
# UPDATE: writes drainage to DA["drainage_area_m2"] (and reads from SP["drainage_area_m2"]
#         if present, otherwise falls back to SP["drainage_area"])
# ============================================================================

import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Define file paths
ext = ".csv"
cropped_file    = f"{base}_cropped_hydro{ext}"
streamline_file = f"{base}_streamline{ext}"

design_path = os.path.join(path, cropped_file)
stream_path = os.path.join(path, streamline_file)

# Load (prefer in-memory if present)
try:
    DA = design_area.copy()
except NameError:
    DA = pd.read_csv(design_path)

try:
    SP = stream_points.copy()
except NameError:
    SP = pd.read_csv(stream_path)

# Ensure numeric
for c in ["x", "y"]:
    if c not in DA.columns or c not in SP.columns:
        raise ValueError(f"Missing '{c}' column. DA cols: {list(DA.columns)} | SP cols: {list(SP.columns)}")
    DA[c] = pd.to_numeric(DA[c], errors="coerce")
    SP[c] = pd.to_numeric(SP[c], errors="coerce")

DA = DA.dropna(subset=["x", "y"]).reset_index(drop=True)
SP = SP.dropna(subset=["x", "y"]).reset_index(drop=True)

# --- Infer grid step on design grid for tolerance ---
def infer_step(vals: np.ndarray) -> float:
    u = np.sort(np.unique(vals))
    diffs = np.diff(u)
    diffs = diffs[diffs > 0]
    if diffs.size == 0:
        return np.nan
    r = np.round(diffs, 6)
    vals_u, counts = np.unique(r, return_counts=True)
    return float(vals_u[np.argmax(counts)])

dx = infer_step(DA["x"].values)
dy = infer_step(DA["y"].values)
candidates = [v for v in [dx, dy] if np.isfinite(v) and v > 0]
grid_step = min(candidates) if candidates else 1.0
tol = 0.51 * grid_step

# --- Compute box-to-box affine transform: (x', y') = (sx*x + tx, sy*y + ty)
xmin_d, xmax_d = float(DA["x"].min()), float(DA["x"].max())
ymin_d, ymax_d = float(DA["y"].min()), float(DA["y"].max())
xmin_s, xmax_s = float(SP["x"].min()), float(SP["x"].max())
ymin_s, ymax_s = float(SP["y"].min()), float(SP["y"].max())

sx = (xmax_d - xmin_d) / (xmax_s - xmin_s) if xmax_s != xmin_s else 1.0
sy = (ymax_d - ymin_d) / (ymax_s - ymin_s) if ymax_s != ymin_s else 1.0
tx = xmin_d - sx * xmin_s
ty = ymin_d - sy * ymin_s

# Transform stream coords into design frame
SP_aligned = SP.copy()
SP_aligned["x_aligned"] = sx * SP["x"] + tx
SP_aligned["y_aligned"] = sy * SP["y"] + ty

# --- Quick sanity check of alignment ---
def rng(a):
    return f"{a.min():.6f} .. {a.max():.6f}"

print("=== ALIGNMENT REPORT ===")
print(f"Design X range: {rng(DA['x'].values)}")
print(f"Stream  X range: {rng(SP['x'].values)}  -> Aligned: {rng(SP_aligned['x_aligned'].values)}")
print(f"Design Y range: {rng(DA['y'].values)}")
print(f"Stream  Y range: {rng(SP['y'].values)}  -> Aligned: {rng(SP_aligned['y_aligned'].values)}")
print(f"sx={sx:.9f}, tx={tx:.6f} | sy={sy:.9f}, ty={ty:.6f}")
print(f"Inferred grid_step={grid_step:.6f}, tol={tol:.6f}")

# --- Nearest-neighbor join in design frame ---
tree = cKDTree(DA[["x", "y"]].to_numpy())
dist, idx = tree.query(SP_aligned[["x_aligned", "y_aligned"]].to_numpy(), k=1)

match_mask = dist <= tol
SPm = SP.loc[match_mask].copy()
SPm_idx = idx[match_mask]

# --- Drainage column mapping (READ from SP, WRITE to DA["drainage_area_m2"]) ---
DRAIN_OUT = "drainage_area_m2"
if "drainage_area_m2" in SPm.columns:
    DRAIN_IN = "drainage_area_m2"
elif "drainage_area" in SPm.columns:
    DRAIN_IN = "drainage_area"
else:
    raise ValueError(f"Stream points missing drainage area column. Have: {list(SP.columns)}")

# If multiple aligned stream points map to the same design index, keep the one with max drainage
updated_rows = 0
skipped_due_to_water = 0

if not SPm.empty:
    tmp = SPm.copy()
    tmp["DA_index"] = SPm_idx

    # Keep max drainage per DA_index
    tmp.sort_values(DRAIN_IN, ascending=False, inplace=True)
    tmp = tmp.drop_duplicates(subset=["DA_index"], keep="first")

    # Ensure required columns exist on DA
    for col in [DRAIN_OUT, "slope", "width_m", "depth_m", "percentile"]:
        if col not in DA.columns:
            DA[col] = np.nan

    # Ensure required columns exist on tmp (safety)
    for col in ["slope", "width_m", "depth_m", "percentile"]:
        if col not in tmp.columns:
            tmp[col] = np.nan

    # Write metrics (NOTE drainage goes to drainage_area_m2)
    DA.loc[tmp["DA_index"], [DRAIN_OUT, "slope", "width_m", "depth_m", "percentile"]] = \
        tmp[[DRAIN_IN, "slope", "width_m", "depth_m", "percentile"]].to_numpy()

    updated_rows = len(tmp)


    # ---- Label update (ONLY upgrade land -> streamline; never touch water) ----
    if "streamline" not in DA.columns:
        DA["streamline"] = pd.Series(pd.NA, index=DA.index, dtype="object")
    else:
        DA["streamline"] = DA["streamline"].astype("object")
    
    idx_arr = tmp["DA_index"].to_numpy()
    cur = DA.loc[idx_arr, "streamline"]
    cur_lc = cur.astype(str).str.strip().str.lower()
    
    is_water = cur_lc.eq("water")
    is_land  = cur_lc.eq("land")
    
    allow = is_land | cur.isna()
    safe_idx = idx_arr[allow.to_numpy()]
    skipped_due_to_water = int(is_water.sum())
    
    DA.loc[safe_idx, "streamline"] = "streamline"

# Save to disk
DA.to_csv(stream_path, index=False)

print("\n Step 7B complete.")
print(f"  Stream points: {len(SP):,} | Matched within tol: {match_mask.sum():,} | Unique design rows updated: {updated_rows:,}")
print(f"  Skipped label upgrades due to water: {skipped_due_to_water:,}")
print(f"  Saved: {stream_path}")

# Peek at a few updated rows
if updated_rows > 0:
    sample_idx = DA.loc[DA["streamline"] == "streamline"].head(10).index
    cols_show = ["x","y","elevation","streamline",DRAIN_OUT,"slope","width_m","depth_m","percentile"]
    cols_show = [c for c in cols_show if c in DA.columns]
    display(DA.loc[sample_idx, cols_show])


In [ ]:
#====================================
#Visualize streamlines
#==============================

ext = ".csv"  # File extension
png = ".png"

streamline_file = f"{base}_streamline{ext}"
heat_file = f"{base}_streamline{png}"

stream_path = os.path.join(path, streamline_file)
csv_path = os.path.join(path, streamline_file)
plot_heatmap_path = os.path.join(path, heat_file)
# ----------------------------
# Load + clean
# ----------------------------
df = pd.read_csv(csv_path, low_memory=False)

# If the file has extra columns, keep only the ones we need
needed = ["x", "y", "percentile"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing columns: {missing}\nFound columns: {list(df.columns)}")

df = df[needed].copy()
df["x"] = pd.to_numeric(df["x"], errors="coerce")
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df["percentile"] = pd.to_numeric(df["percentile"], errors="coerce")
df = df.dropna(subset=["x", "y", "percentile"])

# ----------------------------
# Infer grid spacing (UTM meters)
# ----------------------------
xs = np.sort(df["x"].unique())
ys = np.sort(df["y"].unique())
if len(xs) < 2 or len(ys) < 2:
    raise ValueError("Not enough unique x/y values to form a grid.")

dx = float(np.median(np.diff(xs)))
dy = float(np.median(np.diff(ys)))

xs = np.sort(df["x"].unique())
ys = np.sort(df["y"].unique())

dx = float(np.median(np.diff(xs)))
dy = float(np.median(np.diff(ys)))

print("len(xs) =", len(xs))
print("range/dx =", (xs.max() - xs.min())/dx)
print("largest x gaps:", np.sort(np.diff(xs))[-10:])  # if you see a giant jump, that's it




x0, y0 = xs[0], ys[0]
nx, ny = len(xs), len(ys)

print(f"Loaded: {len(df):,} rows")
print(f"Grid: {ny} rows {nx} cols")
print(f"Spacing: {dy:.3f} m")





# ----------------------------
# Map points to grid indices + aggregate
# ----------------------------
ix = np.rint((df["x"].to_numpy() - x0) / dx).astype(np.int64)
iy = np.rint((df["y"].to_numpy() - y0) / dy).astype(np.int64)
p  = df["percentile"].to_numpy(dtype=np.float64)

valid = (ix >= 0) & (ix < nx) & (iy >= 0) & (iy < ny) & np.isfinite(p)
ix, iy, p = ix[valid], iy[valid], p[valid]

sum_p = np.zeros((ny, nx), dtype=np.float64)
cnt_p = np.zeros((ny, nx), dtype=np.uint32)

np.add.at(sum_p, (iy, ix), p)
np.add.at(cnt_p, (iy, ix), 1)

P = np.full((ny, nx), np.nan, dtype=np.float64)
mask = cnt_p > 0
P[mask] = sum_p[mask] / cnt_p[mask]

# Optional: if percentile is 0..1 and you want 0..100
# P *= 100.0

col_has = (cnt_p.sum(axis=0) > 0)
row_has = (cnt_p.sum(axis=1) > 0)

print("Cols with any data:", col_has.sum(), "/", col_has.size)
print("Rows with any data:", row_has.sum(), "/", row_has.size)

# Find first/last empty runs
empty_cols = np.where(~col_has)[0]
if empty_cols.size:
    print("Empty col index range(s):", empty_cols.min(), "to", empty_cols.max())



# ----------------------------
# Plot
# ----------------------------
extent = [xs.min() - dx/2, xs.max() + dx/2, ys.min() - dy/2, ys.max() + dy/2]

fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(P, origin="lower", extent=extent, aspect="auto")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Percentile")

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title(f"{base}_streamline.csv Percentile heatmap")

plt.tight_layout()
plt.savefig(plot_heatmap_path, dpi=200, bbox_inches='tight')
plt.show()


# slope

In [ ]:
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree

def utm_epsg_from_lonlat(lon, lat):
    zone = int(np.floor((lon + 180) / 6) + 1)
    return (32600 + zone) if lat >= 0 else (32700 + zone)

def add_xy_meters(df, x="x", y="y"):
    """
    If x/y look like lon/lat, project to UTM meters as x_m/y_m.
    If x/y already look projected, just copy to x_m/y_m.
    """
    df = df.copy()
    x_max = df[x].abs().max()
    y_max = df[y].abs().max()

    looks_geographic = (x_max <= 180) and (y_max <= 90)
    if looks_geographic:
        lon0 = float(df[x].median())
        lat0 = float(df[y].median())
        epsg = utm_epsg_from_lonlat(lon0, lat0)
        tf = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        xm, ym = tf.transform(df[x].to_numpy(), df[y].to_numpy())
        df["x_m"] = xm
        df["y_m"] = ym
        df["proj_epsg"] = epsg
        print(f"Projected lon/lat -> UTM meters (EPSG:{epsg})")
    else:
        df["x_m"] = df[x].to_numpy()
        df["y_m"] = df[y].to_numpy()
        df["proj_epsg"] = np.nan
        print("x/y already look projected; using them as meters.")

    return df

def calculate_slope_grid_meters(df, x="x_m", y="y_m", z="elevation"):
    """
    Grid-based slope using np.gradient on a true raster grid (meters).
    Requires rows  unique_x * unique_y (after rounding/snap, if needed).
    """
    df = df.copy()

    # snap to a grid if coordinates are *almost* gridded
    # (round to mm; adjust if your data is noisier)
    df[x] = np.round(df[x].astype(float), 3)
    df[y] = np.round(df[y].astype(float), 3)

    xs = np.sort(df[x].unique())
    ys = np.sort(df[y].unique())

    # infer spacing
    dx = float(np.median(np.diff(xs))) if len(xs) > 1 else 1.0
    dy = float(np.median(np.diff(ys))) if len(ys) > 1 else 1.0
    if dx <= 0 or dy <= 0:
        raise ValueError(f"Bad grid spacing after snap: dx={dx}, dy={dy}")

    expected = len(xs) * len(ys)
    if expected > len(df) * 1.2:
        raise ValueError(
            f"Not a full grid for pivot (expected {expected:,} cells from {len(xs):,}x{len(ys):,}, "
            f"but only {len(df):,} rows). Use kNN slope instead."
        )

    Z = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean").reindex(index=ys, columns=xs).values

    dZ_dy, dZ_dx = np.gradient(Z, dy, dx)
    slope = np.degrees(np.arctan(np.sqrt(dZ_dx**2 + dZ_dy**2)))

    XX, YY = np.meshgrid(xs, ys)
    slope_flat = pd.DataFrame({x: XX.ravel(), y: YY.ravel(), "slope_degrees": slope.ravel()})
    out = df.merge(slope_flat, on=[x, y], how="left")
    out["slope_degrees"] = out["slope_degrees"].fillna(out["slope_degrees"].median())
    return out

def calculate_slope_knn_plane_meters(df, x="x_m", y="y_m", z="elevation", k=20):
    """
    Per-point slope from local best-fit plane using k nearest neighbors (meters).
    Works for scattered points.
    """
    df = df.copy()
    pts = df[[x, y]].to_numpy(dtype=float)
    zz  = df[z].to_numpy(dtype=float)

    tree = cKDTree(pts)
    _, idx = tree.query(pts, k=k)

    x0 = pts[:, 0][:, None]
    y0 = pts[:, 1][:, None]
    Xn = pts[idx, 0] - x0
    Yn = pts[idx, 1] - y0
    Zn = zz[idx] - zz[:, None]

    A00 = np.sum(Xn * Xn, axis=1)
    A01 = np.sum(Xn * Yn, axis=1)
    A11 = np.sum(Yn * Yn, axis=1)
    B0  = np.sum(Xn * Zn, axis=1)
    B1  = np.sum(Yn * Zn, axis=1)

    det = A00 * A11 - A01 * A01
    a = np.where(det != 0, (B0 * A11 - B1 * A01) / det, np.nan)
    b = np.where(det != 0, (B1 * A00 - B0 * A01) / det, np.nan)

    df["slope_degrees"] = np.degrees(np.arctan(np.sqrt(a*a + b*b)))
    df["slope_degrees"] = df["slope_degrees"].fillna(df["slope_degrees"].median())
    return df

def calculate_slope_auto(df):
    """
    Project to meters if needed, then:
    - use grid slope if its a real grid
    - otherwise fall back to kNN plane-fit in meters
    """
    df = add_xy_meters(df, x="x", y="y")

    # try grid method first
    try:
        return calculate_slope_grid_meters(df, x="x_m", y="y_m", z="elevation")
    except Exception as e:
        print("Grid slope not applicable -> using kNN slope. Reason:", e)
        return calculate_slope_knn_plane_meters(df, x="x_m", y="y_m", z="elevation", k=20)


In [ ]:
def add_traversable_column(df, slope_col='slope_degrees', max_slope_deg=45.0, label_col='label'):
    """
    Add a boolean column indicating if agents can traverse this cell.

    Beavers can traverse cells if:
    - Slope is <= max_slope_deg, OR
    - The cell is labeled as 'water' (beavers can swim)

    Parameters
    ----------
    df : DataFrame
        Must contain slope column and label column
    slope_col : str
        Name of slope column
    max_slope_deg : float
        Maximum slope agents can traverse (degrees)
    label_col : str
        Name of label column that contains 'water' values

    Returns
    -------
    DataFrame
        With added 'traversable' column (True/False)
    """
    # Cell is traversable if slope is gentle enough OR if it's water
    df['traversable'] = (df[slope_col] <= max_slope_deg) | (df[label_col] == 'water')
    return df

In [ ]:
# ============================================================================
# Define file paths
# ============================================================================
#os.path.join(path, file_name)
# Define file paths
ext = ".csv"  # File extension
streamline_file = f"{base}_streamline{ext}"
traversible = f"{base}_slopes{ext}"

stream_path = os.path.join(path, streamline_file)
slope_path = os.path.join(path, traversible)




In [ ]:
df = pd.read_csv(stream_path, low_memory=False)

# numeric safety
for c in ["x","y","elevation"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["x","y","elevation"]).copy()

if "label" not in df.columns:
    df["label"] = "land"

print("Calculating slope...")
df = calculate_slope_auto(df)

max_slope_degrees = 50.0
df = add_traversable_column(df, max_slope_deg=max_slope_degrees, label_col="label")

df.to_csv(slope_path, index=False)
print("Saved:", slope_path)

print(df["slope_degrees"].describe())

In [ ]:
def calculate_slope_knn_plane(df, x="x", y="y", z="elevation", k=12):
    """
    Per-point slope from a local best-fit plane using k nearest neighbors.
    If x/y look geographic, project to local UTM meters first so the slope
    denominator is in meters rather than degrees.
    """
    df = df.copy()

    if x not in df.columns or y not in df.columns or z not in df.columns:
        raise ValueError(f"Missing required columns for slope: {[x, y, z]}")

    work = df[[x, y, z]].copy()
    work[x] = pd.to_numeric(work[x], errors="coerce")
    work[y] = pd.to_numeric(work[y], errors="coerce")
    work[z] = pd.to_numeric(work[z], errors="coerce")
    valid = work[[x, y, z]].notna().all(axis=1)

    if valid.sum() < max(3, k):
        raise ValueError(f"Need at least {max(3, k)} valid points for kNN slope; found {int(valid.sum())}.")

    work_valid = work.loc[valid].copy()

    x_max = work_valid[x].abs().max()
    y_max = work_valid[y].abs().max()
    looks_geographic = (x_max <= 180) and (y_max <= 90)

    if looks_geographic:
        lon0 = float(work_valid[x].median())
        lat0 = float(work_valid[y].median())
        epsg = utm_epsg_from_lonlat(lon0, lat0)
        tf = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        xm, ym = tf.transform(work_valid[x].to_numpy(), work_valid[y].to_numpy())
        pts = np.column_stack([xm, ym]).astype(float)
        print(f"kNN slope: projected {x}/{y} to UTM meters (EPSG:{epsg}) before plane fit.")
    else:
        pts = work_valid[[x, y]].to_numpy(dtype=float)
        print(f"kNN slope: using {x}/{y} as projected map units.")

    zz = work_valid[z].to_numpy(dtype=float)
    k_eff = min(int(k), len(work_valid))
    if k_eff < 3:
        raise ValueError("Not enough valid points after filtering for plane fit.")

    tree = cKDTree(pts)
    _, idx = tree.query(pts, k=k_eff)
    if k_eff == 1:
        idx = idx[:, None]

    x0 = pts[:, 0][:, None]
    y0 = pts[:, 1][:, None]

    Xn = pts[idx, 0] - x0
    Yn = pts[idx, 1] - y0
    Zn = zz[idx] - zz[:, None]

    A00 = np.sum(Xn * Xn, axis=1)
    A01 = np.sum(Xn * Yn, axis=1)
    A11 = np.sum(Yn * Yn, axis=1)
    B0 = np.sum(Xn * Zn, axis=1)
    B1 = np.sum(Yn * Zn, axis=1)

    det = A00 * A11 - A01 * A01
    a = np.where(det != 0, (B0 * A11 - B1 * A01) / det, np.nan)
    b = np.where(det != 0, (B1 * A00 - B0 * A01) / det, np.nan)

    slope = np.degrees(np.arctan(np.sqrt(a * a + b * b)))

    df["slope_degrees"] = np.nan
    df.loc[valid, "slope_degrees"] = slope
    df["slope_degrees"] = df["slope_degrees"].fillna(df["slope_degrees"].median())
    return df


In [ ]:
# ============================================================================
# Execute
# ============================================================================

df = pd.read_csv(stream_path, low_memory=False)

# Ensure numeric
for c in ["x", "y", "elevation"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["x", "y", "elevation"]).copy()

# Make sure label exists (so traversable logic works)
if "label" not in df.columns:
    df["label"] = "land"

# Print a grid sanity check
ux = df["x"].nunique()
uy = df["y"].nunique()
grid_cells = ux * uy
print(f"rows: {len(df):,} | unique x: {ux:,} | unique y: {uy:,} | unique_x*unique_y: {grid_cells:,}")

looks_full_grid = (grid_cells == len(df))
if looks_full_grid:
    print("Calculating slope with grid-based gradient in meters (full raster grid detected)...")
else:
    print("Calculating slope with meter-aware fallback (scattered-safe)...")

df = calculate_slope_auto(df)

# ---- Traversability ----
max_slope_degrees = 20.0
df = add_traversable_column(df, max_slope_deg=max_slope_degrees, label_col="label")

# ---- Save ----
df.to_csv(slope_path, index=False)
print("Saved:", slope_path)

print("\nSlope statistics:")
print(f"  Min slope: {df['slope_degrees'].min():.2f}")
print(f"  Max slope: {df['slope_degrees'].max():.2f}")
print(f"  Mean slope: {df['slope_degrees'].mean():.2f}")
print(f"  Median slope: {df['slope_degrees'].median():.2f}")

print(f"\nTraversability ({max_slope_degrees}):")
print(f"  Traversable cells: {df['traversable'].sum():,} ({df['traversable'].mean()*100:.1f}%)")
print(f"  Too steep: {(~df['traversable']).sum():,} ({(~df['traversable']).mean()*100:.1f}%)")

# ---- Plot (scatter, not pivot) ----
plt.figure(figsize=(10, 7))
sc = plt.scatter(df["x"], df["y"], c=df["slope_degrees"], s=2)
plt.colorbar(sc, label="Slope (degrees)")
plt.title("Slope (meter-aware calculation)")
plt.xlabel("x"); plt.ylabel("y")
plt.tight_layout()
plt.savefig(os.path.join(path, f"{base}_slope_scatter.png"), dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 7))
sc2 = plt.scatter(df["x"], df["y"], c=df["traversable"].astype(int), s=2)
plt.colorbar(sc2, label="Traversable (0/1)")
plt.title("Traversability (slope threshold plus water)")
plt.xlabel("x"); plt.ylabel("y")
plt.tight_layout()

plt.savefig(os.path.join(path, f"{base}_traversable_scatter.png"), dpi=200, bbox_inches="tight")
plt.show()


# Import USGS imagery

## Topo data

In [ ]:
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import xy
from rasterio.features import rasterize
from rasterio.transform import from_origin

from pyproj import CRS, Transformer

# Image processing
from skimage.morphology import skeletonize, remove_small_objects, binary_opening, disk
from skimage.measure import label

# Graph traversal for skeleton -> ordered line
import networkx as nx
from shapely.geometry import LineString
import geopandas as gpd


In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def bbox_from_corners(NW, SW, SE, NE):
    lons = [NW[0], SW[0], SE[0], NE[0]]
    lats = [NW[1], SW[1], SE[1], NE[1]]
    left, right = min(lons), max(lons)
    bottom, top = min(lats), max(lats)
    return left, bottom, right, top

def is_rgb(arr):
    return arr.ndim == 3 and arr.shape[0] >= 3


from matplotlib.colors import rgb_to_hsv
import numpy as np

def extract_blue_water_mask(
    rgb,
    hue_min=0.50,
    hue_max=0.69,
    sat_min=0.05,          # was 0.12
    val_min=0.40,          # keep pale but visible blues
    white_floor=0.90,      # slightly lower than 0.92
    white_tolerance=0.08,  # allow off-white paper
    blue_margin=0.02,      # gentler than +0.05
):
    """
    Detect blue/cyan hydrology ink on a white topo background.
    rgb shape: (bands, H, W), RGB order, 0-255.
    """
    img = np.moveaxis(rgb[:3], 0, -1).astype(np.float32) / 255.0
    hsv = rgb_to_hsv(img)

    h = hsv[..., 0]
    s = hsv[..., 1]
    v = hsv[..., 2]

    r = img[..., 0]
    g = img[..., 1]
    b = img[..., 2]

    near_white = (
        (r >= white_floor) & (g >= white_floor) & (b >= white_floor) &
        (np.abs(r - g) <= white_tolerance) &
        (np.abs(r - b) <= white_tolerance) &
        (np.abs(g - b) <= white_tolerance)
    )

    # catches both darker blue stream lines and pale blue pond/water fills
    blueish = (
        (b > r + blue_margin) &
        (b >= g)
    )

    # rejects gray/black map collar / bounds
    not_dark_gray = ~((v < 0.45) & (s < 0.12))

    mask = (
        (h >= hue_min) & (h <= hue_max) &
        (s >= sat_min) &
        (v >= val_min) &
        blueish &
        (~near_white) &
        not_dark_gray
    )

    return mask


def skeleton_to_linestrings(skel_bool, transform_affine, crs):
    """
    Convert a skeletonized binary raster into one or more LineStrings
    by building an 8-neighbor graph over skeleton pixels and extracting a main path
    for each connected component.
    """
    lbl = label(skel_bool, connectivity=2)
    lines = []

    for comp_id in range(1, lbl.max() + 1):
        pts = np.column_stack(np.where(lbl == comp_id))  # rows, cols
        if pts.shape[0] < 10:
            continue

        G = nx.Graph()
        ptset = set(map(tuple, pts))
        for r, c in ptset:
            G.add_node((r, c))
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    if dr == 0 and dc == 0:
                        continue
                    nb = (r + dr, c + dc)
                    if nb in ptset:
                        G.add_edge((r, c), nb)

        deg = dict(G.degree())
        endpoints = [n for n, d in deg.items() if d == 1]

        if len(endpoints) >= 2:
            a = endpoints[0]
            lengths = nx.single_source_shortest_path_length(G, a)
            b = max(endpoints, key=lambda e: lengths.get(e, -1))

            lengths2 = nx.single_source_shortest_path_length(G, b)
            c = max(endpoints, key=lambda e: lengths2.get(e, -1))

            path_nodes = nx.shortest_path(G, b, c)
        else:
            start = next(iter(G.nodes))
            path_nodes = list(nx.dfs_preorder_nodes(G, source=start))

        coords = []
        for r, c in path_nodes:
            x, y = xy(transform_affine, r, c, offset="center")
            coords.append((x, y))

        if len(coords) >= 2:
            lines.append(LineString(coords))

    if not lines:
        return gpd.GeoDataFrame(geometry=[], crs=crs)

    return gpd.GeoDataFrame(geometry=lines, crs=crs)

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from affine import Affine

# -----------------------------
# 2) Read slope CSV and rebuild grid
# -----------------------------
slope_df = pd.read_csv(slope_path)

# Adjust these if your column names differ
x_col = "x"
y_col = "y"
slope_col = "slope_degrees"

required = [x_col, y_col, slope_col]
missing = [c for c in required if c not in slope_df.columns]
if missing:
    raise ValueError(f"Slope CSV is missing required columns: {missing}")

# Unique sorted coordinates
x_unique = np.sort(slope_df[x_col].unique())
y_unique = np.sort(slope_df[y_col].unique())

# Grid size
width = len(x_unique)
height = len(y_unique)

if width < 2 or height < 2:
    raise ValueError("Slope CSV does not define a 2D grid.")

# Infer cell size
dx = np.median(np.diff(x_unique))
dy = np.median(np.diff(y_unique))

# Check regular spacing
if not np.allclose(np.diff(x_unique), dx):
    raise ValueError("X coordinates are not regularly spaced; CSV is not a regular grid.")
if not np.allclose(np.diff(y_unique), dy):
    raise ValueError("Y coordinates are not regularly spaced; CSV is not a regular grid.")

# Build empty raster
slope_array = np.full((height, width), np.nan, dtype=float)

# Map x/y values to column/row indices
x_to_col = {x: i for i, x in enumerate(x_unique)}

# Raster rows usually run top -> bottom, so reverse y
y_desc = y_unique[::-1]
y_to_row = {y: i for i, y in enumerate(y_desc)}

# Fill raster
for _, row in slope_df.iterrows():
    c = x_to_col[row[x_col]]
    r = y_to_row[row[y_col]]
    slope_array[r, c] = row[slope_col]

slope_shape = slope_array.shape

# Build affine transform assuming x/y are cell centers
xmin = x_unique.min()
ymax = y_unique.max()

slope_transform = (
    Affine.translation(xmin - dx / 2, ymax + dy / 2)
    * Affine.scale(dx, -dy)
)

# Recover CRS if the notebook was rerun out of order
if "dem_crs" not in globals() or dem_crs is None:
    crs_candidates = [globals().get(name) for name in ("dem_fp", "tif_path", "cropped_fp", "aligned_path", "topo_tif")]
    crs_candidates = [p for p in crs_candidates if p]

    dem_crs = None
    for candidate in crs_candidates:
        try:
            with rasterio.open(candidate) as src:
                if src.crs is not None:
                    dem_crs = src.crs
                    print(f"Recovered dem_crs from: {candidate}")
                    break
        except Exception:
            pass

    if dem_crs is None:
        raise ValueError(
            "dem_crs is not defined and could not be recovered from available raster paths. "
            "Run the DEM-loading/alignment cell first."
        )

slope_crs = dem_crs
print("slope_crs:", slope_crs)


In [ ]:
topo_files = sorted([
    fp for fp in glob.glob(os.path.join(topo_path, "*.tif"))
    if not os.path.basename(fp).lower().endswith("_cropped.tif")
    and "topo_mosaic" not in os.path.basename(fp).lower()
    and "aligned_to_dem_grid" not in os.path.basename(fp).lower()
    and "_rect" not in os.path.basename(fp).lower()
    and "stream_mask" not in os.path.basename(fp).lower()  # <-- add this
])

In [ ]:
if "dem_crs" not in globals() or dem_crs is None:
    dem_candidate = globals().get("dem_fp") or globals().get("tif_path")
    if dem_candidate:
        with rasterio.open(dem_candidate) as dem:
            dem_crs = dem.crs
            DATA_EPSG = dem_crs.to_epsg()
        print("Loaded DEM CRS from:", dem_candidate)
        print("DEM CRS:", dem_crs)

from pathlib import Path
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds, calculate_default_transform
from rasterio.windows import from_bounds
from rasterio.merge import merge
import glob

# Recover DEM path if the notebook was rerun out of order.
if "dem_fp" not in globals() or not dem_fp:
    dem_candidates = []

    for name in ("tif_path", "cropped_fp", "aligned_path"):
        candidate = globals().get(name)
        if candidate:
            dem_candidates.append(candidate)

    tif_paths_candidate = globals().get("tif_paths")
    if tif_paths_candidate:
        try:
            dem_candidates.extend([p for p in tif_paths_candidate if p])
        except Exception:
            pass

    dem_fp = None
    for candidate in dem_candidates:
        try:
            with rasterio.open(candidate) as src:
                if src.crs is not None:
                    dem_fp = candidate
                    if "dem_crs" not in globals() or dem_crs is None:
                        dem_crs = src.crs
                    print("Recovered dem_fp from:", candidate)
                    break
        except Exception:
            pass

    if dem_fp is None:
        raise ValueError(
            "dem_fp is not defined and could not be recovered from available raster paths. "
            "Run the DEM-loading cell first."
        )

if "dem_crs" not in globals() or dem_crs is None:
    with rasterio.open(dem_fp) as dem:
        dem_crs = dem.crs
        DATA_EPSG = dem_crs.to_epsg()
    print("Loaded DEM CRS from:", dem_fp)
    print("DEM CRS:", dem_crs)

# Re-resolve topo input if a stale path survived a rerun.
if "topo_tif" not in globals() or not topo_tif or not os.path.exists(topo_tif):
    tiff_files = sorted(glob.glob(os.path.join(topo_path, "*.tiff")))
    for src_fp in tiff_files:
        dst_fp = os.path.splitext(src_fp)[0] + ".tif"
        if os.path.exists(dst_fp):
            continue
        with rasterio.open(src_fp) as src:
            meta = src.meta.copy()
            with rasterio.open(dst_fp, "w", **meta) as dst:
                dst.write(src.read())

    topo_files = sorted([
        fp for fp in glob.glob(os.path.join(topo_path, "*.tif"))
        if not os.path.basename(fp).lower().endswith("_cropped.tif")
        and "topo_mosaic" not in os.path.basename(fp).lower()
        and "aligned_to_dem_grid" not in os.path.basename(fp).lower()
        and "_rect" not in os.path.basename(fp).lower()
    ])

    if len(topo_files) == 0:
        raise FileNotFoundError(f"No .tif files found in: {topo_path}")

    if len(topo_files) == 1:
        topo_tif = topo_files[0]
    else:
        topo_merged_path = os.path.join(topo_path, f"{file_name}_Topo_mosaic.tif")

        # Reproject each rotated tile to rectilinear before merging
        reprojected = []
        tmp_paths = []
        for fp in topo_files:
            with rasterio.open(fp) as src:
                if src.transform.is_rectilinear:
                    reprojected.append(rasterio.open(fp))
                    tmp_paths.append(None)
                else:
                    transform, width, height = calculate_default_transform(
                        src.crs, src.crs, src.width, src.height, *src.bounds
                    )
                    meta = src.meta.copy()
                    meta.update({"transform": transform, "width": width, "height": height})
                    tmp_fp = fp.replace(".tif", "_rect.tif")
                    with rasterio.open(tmp_fp, "w", **meta) as dst:
                        for i in range(1, src.count + 1):
                            reproject(
                                source=rasterio.band(src, i),
                                destination=rasterio.band(dst, i),
                                src_transform=src.transform,
                                src_crs=src.crs,
                                dst_transform=transform,
                                dst_crs=src.crs,
                                resampling=Resampling.nearest,
                            )
                    reprojected.append(rasterio.open(tmp_fp))
                    tmp_paths.append(tmp_fp)

        try:
            # Normalize all tiles to the minimum shared band count
            min_bands = min(src.count for src in reprojected)
            mosaic, out_transform = merge(reprojected, indexes=list(range(1, min_bands + 1)))
            out_meta = reprojected[0].meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": out_transform,
                "count": min_bands
            })
            with rasterio.open(topo_merged_path, "w", **out_meta) as dest:
                dest.write(mosaic)
            topo_tif = topo_merged_path
        finally:
            for src in reprojected:
                src.close()
            for tmp in tmp_paths:
                if tmp and os.path.exists(tmp):
                    os.remove(tmp)

    print("Resolved topo_tif:", topo_tif)

if "align_to_dem_grid" not in globals():
    def build_reference_grid(
        dem_fp,
        bbox_lonlat=None,
        target_crs=None,
        target_transform=None,
        target_shape=None
    ):
        if target_crs is not None and target_transform is not None and target_shape is not None:
            return {
                "crs": target_crs,
                "transform": target_transform,
                "height": int(target_shape[0]),
                "width": int(target_shape[1]),
                "source": "existing_dem_grid"
            }

        with rasterio.open(dem_fp) as dem:
            if bbox_lonlat is None:
                return {
                    "crs": dem.crs,
                    "transform": dem.transform,
                    "height": dem.height,
                    "width": dem.width,
                    "source": "full_dem"
                }

            left_b, bottom_b, right_b, top_b = bbox_lonlat
            crop_left, crop_bottom, crop_right, crop_top = transform_bounds(
                "EPSG:4326", dem.crs, left_b, bottom_b, right_b, top_b, densify_pts=21
            )
            clip_left = max(crop_left, dem.bounds.left)
            clip_bottom = max(crop_bottom, dem.bounds.bottom)
            clip_right = min(crop_right, dem.bounds.right)
            clip_top = min(crop_top, dem.bounds.top)
            win = from_bounds(
                clip_left, clip_bottom, clip_right, clip_top, transform=dem.transform
            ).round_offsets().round_lengths()

            return {
                "crs": dem.crs,
                "transform": dem.window_transform(win),
                "height": int(win.height),
                "width": int(win.width),
                "source": "dem_window"
            }

    def same_grid(meta, ref_grid):
        return (
            meta.get("crs") == ref_grid["crs"]
            and meta.get("height") == ref_grid["height"]
            and meta.get("width") == ref_grid["width"]
            and meta.get("transform") == ref_grid["transform"]
        )

    def align_raster_to_reference(src_fp, out_fp, ref_grid, resampling=Resampling.nearest, reuse_if_exists=True):
        out_fp = Path(out_fp)
        out_fp.parent.mkdir(parents=True, exist_ok=True)

        if reuse_if_exists and out_fp.exists():
            with rasterio.open(out_fp) as existing:
                existing_meta = existing.meta.copy()
                if same_grid(existing_meta, ref_grid):
                    print(f"Reusing cached aligned raster: {out_fp}")
                    return existing.read(), existing_meta, out_fp

        with rasterio.open(src_fp) as src:
            if src.crs is None:
                raise ValueError(f"Raster has no CRS and cannot be aligned: {src_fp}")

            dst = np.zeros(
                (src.count, ref_grid["height"], ref_grid["width"]),
                dtype=np.dtype(src.dtypes[0])
            )

            for band_idx in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band_idx),
                    destination=dst[band_idx - 1],
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=ref_grid["transform"],
                    dst_crs=ref_grid["crs"],
                    dst_nodata=src.nodata,
                    resampling=resampling,
                )

            meta = src.meta.copy()
            meta.update({
                "driver": "GTiff",
                "crs": ref_grid["crs"],
                "transform": ref_grid["transform"],
                "height": ref_grid["height"],
                "width": ref_grid["width"],
                "count": src.count
            })
            if src.nodata is not None:
                meta["nodata"] = src.nodata

        with rasterio.open(out_fp, "w", **meta) as dst_ds:
            dst_ds.write(dst)

        return dst, meta, out_fp

    def align_to_dem_grid(
        src_fp,
        layer_name,
        out_dir,
        dem_fp,
        bbox_lonlat=None,
        prefer_existing_grid=True,
        resampling=Resampling.nearest,
        reuse_if_exists=True,
    ):
        target_crs = globals().get("slope_crs") if prefer_existing_grid else None
        target_transform = globals().get("slope_transform") if prefer_existing_grid else None
        target_shape = globals().get("slope_shape") if prefer_existing_grid else None

        ref_grid = build_reference_grid(
            dem_fp=dem_fp,
            bbox_lonlat=bbox_lonlat,
            target_crs=target_crs,
            target_transform=target_transform,
            target_shape=target_shape,
        )

        src_path = Path(src_fp)
        out_dir = Path(out_dir)
        out_fp = out_dir / f"{src_path.stem}_{layer_name}_aligned_to_dem_grid.tif"

        aligned_data, aligned_meta, aligned_fp = align_raster_to_reference(
            src_fp=src_fp,
            out_fp=out_fp,
            ref_grid=ref_grid,
            resampling=resampling,
            reuse_if_exists=reuse_if_exists,
        )

        return {
            "data": aligned_data,
            "meta": aligned_meta,
            "path": aligned_fp,
            "ref_grid": ref_grid,
        }

if not all(name in globals() for name in ["left", "bottom", "right", "top"]):
    left, bottom, right, top = bbox_from_corners(NW, SW, SE, NE)

out_dir = Path(topo_path) / "derived"
topo_result = align_to_dem_grid(
    src_fp=topo_tif,
    layer_name="topo",
    out_dir=out_dir,
    dem_fp=dem_fp,
    bbox_lonlat=(left, bottom, right, top),
    prefer_existing_grid=True,
    resampling=Resampling.nearest,
)

aligned = topo_result["data"]
aligned_meta = topo_result["meta"]
aligned_path = topo_result["path"]
crop_path = aligned_path
crop_meta = aligned_meta
topo_crop = aligned


In [ ]:
aligned_path = Path(aligned_path)
if not aligned_path.exists():
    aligned_candidates = sorted(
        Path(out_dir).glob("*_topo_aligned_to_dem_grid.tif"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not aligned_candidates:
        raise FileNotFoundError(f"Aligned topo raster not found: {aligned_path}")
    aligned_path = aligned_candidates[0]
    print("Recovered aligned_path from derived topo raster:", aligned_path)
with rasterio.open(aligned_path) as src:
    aligned = src.read()
    aligned_meta = src.meta.copy()
print("Aligned topo bands:", aligned.shape[0])          # <-- add this
print("Aligned topo CRS:", aligned_meta["crs"])
print("Aligned topo transform:", aligned_meta["transform"])
print("Aligned topo grid (rows, cols):", aligned.shape[1:])

In [ ]:
from skimage.morphology import skeletonize, opening, disk
from skimage.measure import label
from matplotlib.colors import rgb_to_hsv

def load_rgb_aligned_topo(preferred_path=None, search_dir=None):
    candidates = []

    if preferred_path is not None:
        candidates.append(Path(preferred_path))

    if "topo_result" in globals():
        topo_path_candidate = topo_result.get("path")
        if topo_path_candidate is not None:
            candidates.append(Path(topo_path_candidate))

    if search_dir is not None:
        candidates.extend(sorted(
            Path(search_dir).glob("*_topo_aligned_to_dem_grid.tif"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        ))

    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen or (not candidate.exists()):
            continue
        seen.add(candidate)

        with rasterio.open(candidate) as src:
            data = src.read()
            meta = src.meta.copy()
        print(f"Checked aligned topo candidate: {candidate.name} -> {data.shape[0]} band(s)")

        if is_rgb(data):
            return candidate, data, meta

    raise ValueError(
        "Could not find an RGB topo raster aligned to the DEM grid. "
        "The current aligned raster is single-band, so blue-water extraction cannot run on it."
    )


aligned_path, aligned, aligned_meta = load_rgb_aligned_topo(
    preferred_path=globals().get("aligned_path"),
    search_dir=globals().get("out_dir"),
)
print(f"Using aligned topo: {aligned_path}")
print(f"Reloaded aligned: {aligned.shape[0]} bands, shape={aligned.shape[1:]}")


def remove_small_components(mask, min_pixels=8, connectivity=2):
    lbl = label(mask, connectivity=connectivity)
    if lbl.max() == 0:
        return mask.astype(bool)

    counts = np.bincount(lbl.ravel())
    keep = counts >= min_pixels
    keep[0] = False
    return keep[lbl]

# -----------------------------
# 5) Extract full water mask (blue) on aligned DEM grid
# -----------------------------
water_mask = extract_blue_water_mask(aligned)
water_mask = remove_small_components(water_mask, min_pixels=8)

# keep optional centerlines for vector output only
skel = skeletonize(water_mask)
gdf_lines = skeleton_to_linestrings(
    skel,
    aligned_meta["transform"],
    aligned_meta["crs"]
)


base_name = Path(topo_tif).stem

# Save as GeoJSON in slope CRS + also in lon/lat
stream_geojson_slope = out_dir / f"{base_name}_streamlines_slopeCRS.geojson"
gdf_lines.to_file(stream_geojson_slope, driver="GeoJSON")
print("Wrote streamlines (slope CRS):", stream_geojson_slope)
gdf_ll = gdf_lines.to_crs("EPSG:4326")
stream_geojson_ll = out_dir / f"{base_name}_streamlines_lonlat.geojson"
gdf_ll.to_file(stream_geojson_ll, driver="GeoJSON")
print("Wrote streamlines (lon/lat):", stream_geojson_ll)


In [ ]:
# -----------------------------
# 6) Write full water mask onto master CSV
# -----------------------------
stream_mask = water_mask.astype(np.uint8)

base_name = Path(topo_tif).stem

stream_mask_path = out_dir / f"{base_name}_water_mask_on_slope_grid.tif"
mask_meta = {
    "driver": "GTiff",
    "height": stream_mask.shape[0],
    "width": stream_mask.shape[1],
    "count": 1,
    "dtype": "uint8",
    "crs": aligned_meta["crs"],
    "transform": aligned_meta["transform"],
}
with rasterio.open(stream_mask_path, "w", **mask_meta) as dst:
    dst.write(stream_mask, 1)

print("Wrote water mask:", stream_mask_path)

# -----------------------------
# Build grid-center table from stream mask
# -----------------------------
rows, cols = np.indices(stream_mask.shape)
xs, ys = rasterio.transform.xy(aligned_meta["transform"], rows, cols, offset="center")
xs = np.array(xs).ravel()
ys = np.array(ys).ravel()

stream_df = pd.DataFrame({
    "x": xs,
    "y": ys,
    "stream": stream_mask.ravel().astype(np.uint8),   # keep name if downstream expects it
})

# Round coordinates to make merge robust to float precision
stream_df["x_round"] = stream_df["x"].round(3)
stream_df["y_round"] = stream_df["y"].round(3)

# Optional lon/lat from slope CRS
to_ll = Transformer.from_crs(
    CRS.from_user_input(slope_crs),
    CRS.from_epsg(4326),
    always_xy=True
)
lons, lats = to_ll.transform(stream_df["x"].values, stream_df["y"].values)
stream_df["lon"] = lons
stream_df["lat"] = lats

# -----------------------------
# Read master CSV and merge stream labels into it
# -----------------------------
master_df = pd.read_csv(slope_path, low_memory=False)

for c in ["x", "y"]:
    master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

master_df = master_df.dropna(subset=["x", "y"]).copy()
master_df["x_round"] = master_df["x"].round(3)
master_df["y_round"] = master_df["y"].round(3)

master_df = master_df.merge(
    stream_df[["x_round", "y_round", "stream", "lon", "lat"]],
    on=["x_round", "y_round"],
    how="left"
)

# Fill missing stream cells as 0
master_df["stream"] = master_df["stream"].fillna(0).astype(np.uint8)

# Only write lon/lat if they do not already exist or are empty
if "lon" not in master_df.columns:
    master_df["lon"] = master_df["lon_y"]
if "lat" not in master_df.columns:
    master_df["lat"] = master_df["lat_y"]

# If lon/lat already existed, prefer existing values and fill gaps from new values
if "lon_x" in master_df.columns and "lon_y" in master_df.columns:
    master_df["lon"] = master_df["lon_x"].where(master_df["lon_x"].notna(), master_df["lon_y"])
if "lat_x" in master_df.columns and "lat_y" in master_df.columns:
    master_df["lat"] = master_df["lat_x"].where(master_df["lat_x"].notna(), master_df["lat_y"])

# Optional: mark stream cells in label column
if "label" in master_df.columns:
    master_df.loc[master_df["stream"] == 1, "label"] = "water"

# Clean up helper/duplicate columns
drop_cols = [c for c in [
    "x_round", "y_round",
    "lon_x", "lon_y", "lat_x", "lat_y"
] if c in master_df.columns]
master_df = master_df.drop(columns=drop_cols)

# Overwrite the master CSV
master_df.to_csv(slope_path, index=False)
print("Updated master CSV:", slope_path)
print("Stream cells written:", int(master_df["stream"].sum()))

In [ ]:
import matplotlib.pyplot as plt
from rasterio.transform import array_bounds

if "slope_shape" not in globals() or "slope_transform" not in globals():
    raise NameError("slope_shape and slope_transform must be defined before plotting the aligned topo diagnostics.")

if "slope_array" in globals() and slope_array is not None:
    dem_grid = np.asarray(slope_array, dtype=np.float32)
elif "slope_df" in globals() and slope_df is not None:
    _plot_df = slope_df.copy()
elif "slope_path" in globals() and os.path.exists(slope_path):
    _plot_df = pd.read_csv(slope_path, low_memory=False)
else:
    raise NameError("Could not build DEM plot grid. Expected slope_array, slope_df, or a valid slope_path.")

if "dem_grid" not in globals() or dem_grid is None:
    required_cols = ["x", "y", "elevation"]
    missing_cols = [c for c in required_cols if c not in _plot_df.columns]
    if missing_cols:
        raise ValueError(f"Slope CSV is missing required columns for plotting: {missing_cols}")

    for c in required_cols:
        _plot_df[c] = pd.to_numeric(_plot_df[c], errors="coerce")
    _plot_df = _plot_df.dropna(subset=required_cols)

    dem_grid = (
        _plot_df[["x", "y", "elevation"]]
        .pivot_table(index="y", columns="x", values="elevation", aggfunc="first")
        .reindex(index=np.sort(_plot_df["y"].unique())[::-1], columns=np.sort(_plot_df["x"].unique()))
        .to_numpy(dtype=np.float32)
    )

west, south, east, north = array_bounds(slope_shape[0], slope_shape[1], slope_transform)
extent = [west, east, south, north]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

im0 = axes[0].imshow(dem_grid, extent=extent, origin="upper")
axes[0].set_title("DEM")
fig.colorbar(im0, ax=axes[0], label="Elevation (m)")

if "aligned" in globals() and aligned is not None and np.ndim(aligned) == 3 and aligned.shape[0] >= 3:
    rgb = np.moveaxis(aligned[:3], 0, -1)
    axes[1].imshow(rgb, extent=extent, origin="upper")
    axes[1].set_title("Aligned topo")
else:
    axes[1].imshow(dem_grid, extent=extent, origin="upper", cmap="terrain")
    axes[1].set_title("Aligned topo unavailable")

if "blue" in globals() and blue is not None:
    axes[2].imshow(dem_grid, extent=extent, origin="upper")
    axes[2].imshow(np.ma.masked_where(~blue, blue), extent=extent, origin="upper", alpha=0.7)
    axes[2].set_title("Blue stream mask")
else:
    axes[2].imshow(dem_grid, extent=extent, origin="upper", cmap="terrain")
    axes[2].set_title("Blue mask unavailable")

if "skel" in globals() and skel is not None:
    axes[3].imshow(dem_grid, extent=extent, origin="upper")
    axes[3].imshow(np.ma.masked_where(~skel, skel), extent=extent, origin="upper", alpha=0.7)
    if "gdf_lines" in globals() and gdf_lines is not None and not gdf_lines.empty:
        gdf_lines.plot(ax=axes[3], linewidth=1)
    axes[3].set_title("Skeleton and vector lines")
else:
    axes[3].imshow(dem_grid, extent=extent, origin="upper", cmap="terrain")
    axes[3].set_title("Skeleton unavailable")

for ax in axes:
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()


## Import NAIP Imagery

In [ ]:
# Build geographic bbox
left, bottom, right, top = bbox_from_corners(NW, SW, SE, NE)
#naip_fp = os.path.join(naip_path, naip_name)

In [ ]:
# DEM file = the authoritative CRS/resolution for the project
from pathlib import Path
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import from_bounds

dem_fp = tif_path

with rasterio.open(dem_fp) as dem:
    dem_crs = dem.crs
    DATA_EPSG = dem_crs.to_epsg()
    dem_transform = dem.transform
    dem_res = dem.res
    dem_bounds = dem.bounds
    dem_height = dem.height
    dem_width = dem.width

    if dem_crs is None:
        raise ValueError("DEM has no CRS; cannot align the workflow.")

    print("DEM CRS:", dem_crs)
    print("DATA_EPSG:", DATA_EPSG)
    print("DEM bounds:", dem_bounds)
    print("DEM res:", dem_res)

def build_reference_grid(
    dem_fp,
    bbox_lonlat=None,
    target_crs=None,
    target_transform=None,
    target_shape=None
):
    if target_crs is not None and target_transform is not None and target_shape is not None:
        return {
            "crs": target_crs,
            "transform": target_transform,
            "height": int(target_shape[0]),
            "width": int(target_shape[1]),
            "source": "existing_dem_grid"
        }

    with rasterio.open(dem_fp) as dem:
        if bbox_lonlat is None:
            return {
                "crs": dem.crs,
                "transform": dem.transform,
                "height": dem.height,
                "width": dem.width,
                "source": "full_dem"
            }

        left, bottom, right, top = bbox_lonlat
        crop_left, crop_bottom, crop_right, crop_top = transform_bounds(
            "EPSG:4326", dem.crs, left, bottom, right, top, densify_pts=21
        )

        clip_left = max(crop_left, dem.bounds.left)
        clip_bottom = max(crop_bottom, dem.bounds.bottom)
        clip_right = min(crop_right, dem.bounds.right)
        clip_top = min(crop_top, dem.bounds.top)

        if clip_left >= clip_right or clip_bottom >= clip_top:
            raise ValueError("Requested AOI does not overlap the DEM.")

        win = from_bounds(
            clip_left, clip_bottom, clip_right, clip_top, transform=dem.transform
        ).round_offsets().round_lengths()

        if win.width <= 0 or win.height <= 0:
            raise ValueError("DEM crop window is empty after snapping to the DEM grid.")

        return {
            "crs": dem.crs,
            "transform": dem.window_transform(win),
            "height": int(win.height),
            "width": int(win.width),
            "source": "dem_window"
        }

def same_grid(meta, ref_grid):
    return (
        meta.get("crs") == ref_grid["crs"]
        and meta.get("height") == ref_grid["height"]
        and meta.get("width") == ref_grid["width"]
        and meta.get("transform") == ref_grid["transform"]
    )

def align_raster_to_reference(
    src_fp,
    out_fp,
    ref_grid,
    resampling=Resampling.nearest,
    reuse_if_exists=True
):
    out_fp = Path(out_fp)
    out_fp.parent.mkdir(parents=True, exist_ok=True)

    if reuse_if_exists and out_fp.exists():
        with rasterio.open(out_fp) as existing:
            existing_meta = existing.meta.copy()
            if same_grid(existing_meta, ref_grid):
                print(f"Reusing cached aligned raster: {out_fp}")
                return existing.read(), existing_meta, out_fp

    with rasterio.open(src_fp) as src:
        if src.crs is None:
            raise ValueError(f"Raster has no CRS and cannot be aligned: {src_fp}")

        if (
            src.crs == ref_grid["crs"]
            and src.transform == ref_grid["transform"]
            and src.height == ref_grid["height"]
            and src.width == ref_grid["width"]
        ):
            print(f"Source already matches DEM grid: {src_fp}")
            data = src.read()
            meta = src.meta.copy()
            if src_fp != str(out_fp):
                with rasterio.open(out_fp, "w", **meta) as dst_ds:
                    dst_ds.write(data)
            return data, meta, out_fp

        dst = np.zeros(
            (src.count, ref_grid["height"], ref_grid["width"]),
            dtype=np.dtype(src.dtypes[0])
        )

        for band_idx in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, band_idx),
                destination=dst[band_idx - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src.nodata,
                dst_transform=ref_grid["transform"],
                dst_crs=ref_grid["crs"],
                dst_nodata=src.nodata,
                resampling=resampling,
            )

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "crs": ref_grid["crs"],
            "transform": ref_grid["transform"],
            "height": ref_grid["height"],
            "width": ref_grid["width"],
            "count": src.count
        })
        if src.nodata is not None:
            meta["nodata"] = src.nodata

    with rasterio.open(out_fp, "w", **meta) as dst_ds:
        dst_ds.write(dst)

    return dst, meta, out_fp

def align_to_dem_grid(
    src_fp,
    layer_name,
    out_dir,
    dem_fp,
    bbox_lonlat=None,
    prefer_existing_grid=True,
    resampling=Resampling.nearest,
    reuse_if_exists=True,
):
    target_crs = globals().get("slope_crs") if prefer_existing_grid else None
    target_transform = globals().get("slope_transform") if prefer_existing_grid else None
    target_shape = globals().get("slope_shape") if prefer_existing_grid else None

    ref_grid = build_reference_grid(
        dem_fp=dem_fp,
        bbox_lonlat=bbox_lonlat,
        target_crs=target_crs,
        target_transform=target_transform,
        target_shape=target_shape,
    )

    src_path = Path(src_fp)
    out_dir = Path(out_dir)
    out_fp = out_dir / f"{src_path.stem}_{layer_name}_aligned_to_dem_grid.tif"

    aligned_data, aligned_meta, aligned_fp = align_raster_to_reference(
        src_fp=src_fp,
        out_fp=out_fp,
        ref_grid=ref_grid,
        resampling=resampling,
        reuse_if_exists=reuse_if_exists,
    )

    print(f"{layer_name} source:", src_fp)
    print(f"{layer_name} target grid source:", ref_grid["source"])
    print(f"{layer_name} aligned shape:", aligned_data.shape)
    print(f"{layer_name} aligned path:", aligned_fp)

    return {
        "data": aligned_data,
        "meta": aligned_meta,
        "path": aligned_fp,
        "ref_grid": ref_grid,
    }

print("Alignment helper ready: use align_to_dem_grid(...) for future datasets.")


In [ ]:
if "dem_crs" not in globals() or dem_crs is None:
    dem_fp = globals().get("dem_fp", tif_path)
    with rasterio.open(dem_fp) as dem:
        dem_crs = dem.crs
        DATA_EPSG = dem_crs.to_epsg()
    print("Loaded DEM CRS from:", dem_fp)
    print("DEM CRS:", dem_crs)

from pathlib import Path
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.warp import reproject, calculate_default_transform, transform_bounds
import glob


def merge_with_normalized_inputs(filepaths, out_fp, bbox_lonlat=None, resampling=Resampling.nearest):
    src_files = []
    temp_paths = []

    try:
        with rasterio.open(filepaths[0]) as ref_src:
            ref_crs = ref_src.crs
            if ref_crs is None:
                raise ValueError(f"Reference raster has no CRS: {filepaths[0]}")

        merge_bounds = None
        if bbox_lonlat is not None:
            left_b, bottom_b, right_b, top_b = bbox_lonlat
            merge_bounds = transform_bounds(
                "EPSG:4326", ref_crs, left_b, bottom_b, right_b, top_b, densify_pts=21
            )

        for fp in filepaths:
            src = rasterio.open(fp)
            needs_warp = (src.crs != ref_crs) or (not src.transform.is_rectilinear)

            if not needs_warp:
                src_files.append(src)
                temp_paths.append(None)
                continue

            transform, width, height = calculate_default_transform(
                src.crs, ref_crs, src.width, src.height, *src.bounds
            )
            meta = src.meta.copy()
            meta.update({
                "crs": ref_crs,
                "transform": transform,
                "width": width,
                "height": height,
            })
            tmp_fp = os.path.splitext(fp)[0] + "_norm.tif"

            with rasterio.open(tmp_fp, "w", **meta) as dst:
                for band_idx in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, band_idx),
                        destination=rasterio.band(dst, band_idx),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=ref_crs,
                        resampling=resampling,
                    )

            src.close()
            src_files.append(rasterio.open(tmp_fp))
            temp_paths.append(tmp_fp)

        min_bands = min(src.count for src in src_files)
        mosaic, out_transform = merge(
            src_files,
            indexes=list(range(1, min_bands + 1)),
            bounds=merge_bounds,
        )

        out_meta = src_files[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],
            "width": mosaic.shape[2],
            "transform": out_transform,
            "count": min_bands
        })

        with rasterio.open(out_fp, "w", **out_meta) as dest:
            dest.write(mosaic)
    finally:
        for src in src_files:
            src.close()
        for tmp_fp in temp_paths:
            if tmp_fp and os.path.exists(tmp_fp):
                os.remove(tmp_fp)


if not all(name in globals() for name in ["left", "bottom", "right", "top"]):
    left, bottom, right, top = bbox_from_corners(NW, SW, SE, NE)

if "naip_fp" not in globals() or not naip_fp:
    tif_files = globals().get("naip_tif_files")
    if not tif_files:
        tif_files = sorted([
            fp for fp in glob.glob(os.path.join(naip_path, "*.tif"))
            if not os.path.basename(fp).lower().endswith("_cropped.tif")
            and "naip_mosaic" not in os.path.basename(fp).lower()
            and "aligned_to_dem_grid" not in os.path.basename(fp).lower()
            and "_norm" not in os.path.basename(fp).lower()
        ])

    if len(tif_files) == 0:
        raise FileNotFoundError(f"No .tif files found in: {naip_path}")

    if len(tif_files) == 1:
        naip_fp = tif_files[0]
    else:
        naip_merged_path = os.path.join(naip_path, f"{base}_NAIP_mosaic_aoi.tif")
        print("Merging NAIP tiles only over the AOI to avoid huge full-scene mosaics...")
        merge_with_normalized_inputs(
            tif_files,
            naip_merged_path,
            bbox_lonlat=(left, bottom, right, top),
            resampling=Resampling.bilinear,
        )
        naip_fp = naip_merged_path

    print("Resolved naip_fp:", naip_fp)

naip_result = align_to_dem_grid(
    src_fp=naip_fp,
    layer_name="naip",
    out_dir=naip_path,
    dem_fp=dem_fp,
    bbox_lonlat=(left, bottom, right, top),
    prefer_existing_grid=True,
    resampling=Resampling.bilinear,
)

cropped = naip_result["data"]
cropped_meta = naip_result["meta"]
cropped_fp = str(naip_result["path"])


In [ ]:
rgb = np.moveaxis(cropped[:3], 0, -1)

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title("NAIP aligned to DEM grid")
plt.axis("off")
plt.show()

print("Using DEM-grid NAIP raster:", cropped_fp)


In [ ]:
# ============================================================
# INPUTS
# ============================================================
tif_path = cropped_fp
# ============================================================
# LOAD MASTER CSV
# ============================================================
df = pd.read_csv(slope_path)



In [ ]:
# ============================================================
# HELPER: safe divide
# ============================================================
def safe_divide(num, den):
    num = np.asarray(num, dtype="float64")
    den = np.asarray(den, dtype="float64")
    out = np.full(num.shape, np.nan, dtype="float64")
    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]
    return out


In [ ]:
if "dem_crs" not in globals() or dem_crs is None:
    dem_fp = globals().get("dem_fp", tif_path)
    with rasterio.open(dem_fp) as dem:
        dem_crs = dem.crs
        DATA_EPSG = dem_crs.to_epsg()
    print("Loaded DEM CRS from:", dem_fp)
    print("DEM CRS:", dem_crs)

from rasterio.transform import rowcol

with rasterio.open(tif_path) as src:
    print("Raster CRS:", src.crs)
    print("Raster shape:", (src.height, src.width))
    print("Raster band count:", src.count)

    if src.count < 4:
        raise ValueError("The TIFF has fewer than 4 bands.")
    if src.crs != dem_crs:
        raise ValueError(f"Expected DEM-aligned NAIP in {dem_crs}, but found {src.crs}.")

    cols_lower = {c.lower(): c for c in df.columns}
    x_col = cols_lower.get("x")
    y_col = cols_lower.get("y")
    lon_col = next((cols_lower[c] for c in ["longitude", "lon", "long"] if c in cols_lower), None)
    lat_col = next((cols_lower[c] for c in ["latitude", "lat"] if c in cols_lower), None)

    def build_candidate(xs, ys, label):
        xs = np.asarray(xs, dtype="float64")
        ys = np.asarray(ys, dtype="float64")
        finite = np.isfinite(xs) & np.isfinite(ys)
        rows = np.full(xs.shape, -1, dtype=int)
        cols = np.full(xs.shape, -1, dtype=int)
        if finite.any():
            rows_f, cols_f = rowcol(src.transform, xs[finite], ys[finite])
            rows[finite] = np.asarray(rows_f, dtype=int)
            cols[finite] = np.asarray(cols_f, dtype=int)
        inside = finite & (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)
        return {
            "label": label,
            "rows": rows,
            "cols": cols,
            "finite": finite,
            "inside": inside,
            "inside_count": int(inside.sum()),
            "finite_count": int(finite.sum()),
        }

    candidates = []

    if x_col is not None and y_col is not None:
        xy_xs = pd.to_numeric(df[x_col], errors="coerce").to_numpy()
        xy_ys = pd.to_numeric(df[y_col], errors="coerce").to_numpy()
        candidates.append(build_candidate(xy_xs, xy_ys, f"{x_col}/{y_col}"))

    if lon_col is not None and lat_col is not None:
        lons = pd.to_numeric(df[lon_col], errors="coerce").to_numpy()
        lats = pd.to_numeric(df[lat_col], errors="coerce").to_numpy()
        transformed_xs, transformed_ys = Transformer.from_crs("EPSG:4326", dem_crs, always_xy=True).transform(lons, lats)
        candidates.append(build_candidate(transformed_xs, transformed_ys, f"{lon_col}/{lat_col} -> DEM CRS"))

    if not candidates:
        raise ValueError("Could not find usable coordinate columns.")

    coord_choice = max(candidates, key=lambda c: (c["inside_count"], c["finite_count"]))
    valid_xy = coord_choice["finite"]
    inside = coord_choice["inside"]

    print("Candidate coordinate coverage:")
    for candidate in candidates:
        frac = (candidate["inside_count"] / candidate["finite_count"]) if candidate["finite_count"] else np.nan
        print(
            f"  {candidate['label']}: inside={candidate['inside_count']} "
            f"of {candidate['finite_count']} finite rows ({frac:.3f})"
        )
    print("Using coordinates:", coord_choice["label"])

    band_stack = src.read([1, 2, 3, 4]).astype("float64")
    nodata = src.nodata
    if nodata is not None:
        band_stack[band_stack == nodata] = np.nan

    samples = np.full((len(df), 4), np.nan, dtype="float64")
    inside_idx = np.where(inside)[0]
    if inside_idx.size:
        rows = coord_choice["rows"][inside_idx]
        cols = coord_choice["cols"][inside_idx]
        samples[inside_idx] = band_stack[:, rows, cols].T

    print("Rows with finite coordinates:", int(valid_xy.sum()), "of", len(df))
    print("Rows landing on raster cells:", int(inside.sum()), "of", len(df))


In [ ]:
#Validate red, green, blue, and NIR bands

with rasterio.open(tif_path) as src:
    b1 = src.read(1)
    b2 = src.read(2)
    b3 = src.read(3)
    b4 = src.read(4)
    dmask = src.dataset_mask()

print("Fraction band4 == dataset_mask:", np.mean(b4 == dmask))
print("Band4 unique values:", len(np.unique(b4)))
print("dataset_mask unique values:", np.unique(dmask)[:20], "...")

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(b4, cmap="gray")
plt.title("Band 4")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(dmask, cmap="gray")
plt.title("Dataset mask")
plt.axis("off")

plt.subplot(1,3,3)
plt.hist(b4.ravel(), bins=50)
plt.title("Band 4 histogram")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import rasterio

df = pd.read_csv(slope_path)

with rasterio.open(cropped_fp) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Shape:", src.height, src.width)
    print("Transform:", src.transform)
    print("Nodata:", src.nodata)

    if "slope_shape" in globals():
        print("Matches slope grid shape:", (src.height, src.width) == tuple(slope_shape))
    if "slope_transform" in globals():
        print("Matches slope grid transform:", src.transform == slope_transform)


In [ ]:
df["naip_R"] = samples[:, 0]
df["naip_G"] = samples[:, 1]
df["naip_B"] = samples[:, 2]
df["naip_NIR"] = samples[:, 3]


In [ ]:
df["naip_R"] 

In [ ]:
# ============================================================
# CALCULATE GREENNESS / VEGETATION INDICES
# ============================================================
R   = df["naip_R"].to_numpy(dtype="float64")
G   = df["naip_G"].to_numpy(dtype="float64")
B   = df["naip_B"].to_numpy(dtype="float64")
NIR = df["naip_NIR"].to_numpy(dtype="float64")

# NDVI = (NIR - R) / (NIR + R)
df["veg_NDVI"] = safe_divide(NIR - R, NIR + R)

# GNDVI = (NIR - G) / (NIR + G)
df["veg_GNDVI"] = safe_divide(NIR - G, NIR + G)

# Normalized Green = G / (R + G + B)
df["veg_NormG"] = safe_divide(G, R + G + B)

# EVI = 2.5 * (NIR - R) / (NIR + 6R - 7.5B + 1)
df["veg_EVI"] = 2.5 * safe_divide(NIR - R, NIR + 6 * R - 7.5 * B + 1)


In [ ]:
# ============================================================
# SAVE BACK TO MASTER CSV
# ============================================================
df.to_csv(slope_path, index=False)
print("Saved:", slope_path)

# quick check
new_cols = ["naip_R", "naip_G", "naip_B", "naip_NIR",
            "veg_NDVI", "veg_GNDVI", "veg_NormG", "veg_EVI"]
print(df[new_cols].head())

# Validate resolution of DEM

In [ ]:
def estimate_grid_spacing(df, x_col="x", y_col="y"):
    x = pd.to_numeric(df[x_col], errors="coerce").dropna().to_numpy()
    y = pd.to_numeric(df[y_col], errors="coerce").dropna().to_numpy()

    xs = np.unique(x)
    ys = np.unique(y)
    xs.sort()
    ys.sort()

    if len(xs) < 2 or len(ys) < 2:
        raise ValueError("Not enough unique x/y values to estimate spacing.")

    dx = np.median(np.diff(xs))
    dy = np.median(np.diff(ys))

    return dx, dy, len(xs), len(ys)

# Example usage
df = pd.read_csv(csv_path, low_memory=False)
dx, dy, nx, ny = estimate_grid_spacing(df, x_col="x", y_col="y")

print(f"Grid dims: {ny} rows  {nx} cols")
print(f"Estimated spacing: dx={dx:.6f}, dy={dy:.6f}")

# Heuristic: if this is UTM (meters), dx/dy should be ~1.0
if abs(dx) > 1 and abs(dy) > 1:
    print("Looks like projected units (meters). Resolution is in meters.")
else:
    print("Looks like geographic degrees. Convert to meters using latitude if needed.")

import numpy as np
import pandas as pd

FT_PER_M = 3.280839895
M2_PER_ACRE = 4046.8564224



if "res_stats" not in globals() or res_stats is None:
    dx_safe = float(abs(dx)) if "dx" in globals() and dx is not None and np.isfinite(dx) else np.nan
    dy_safe = float(abs(dy)) if "dy" in globals() and dy is not None and np.isfinite(dy) else np.nan

    mean_m = np.nanmean([dx_safe, dy_safe])
    med_m  = np.nanmedian([dx_safe, dy_safe])

    res_stats = {
        "mean_meters": mean_m,
        "mean_feet": mean_m * FT_PER_M if np.isfinite(mean_m) else np.nan,
        "median_meters": med_m,
        "median_feet": med_m * FT_PER_M if np.isfinite(med_m) else np.nan,
    }


def estimate_grid_spacing(df, x_col="x", y_col="y"):
    x = pd.to_numeric(df[x_col], errors="coerce").dropna().to_numpy()
    y = pd.to_numeric(df[y_col], errors="coerce").dropna().to_numpy()

    xs = np.unique(x); xs.sort()
    ys = np.unique(y); ys.sort()

    if len(xs) < 2 or len(ys) < 2:
        raise ValueError("Not enough unique x/y values to estimate spacing.")

    dx = float(np.median(np.diff(xs)))
    dy = float(np.median(np.diff(ys)))

    return dx, dy, len(xs), len(ys), xs.min(), xs.max(), ys.min(), ys.max()

def _meters_per_degree(lat_deg: float):
    """Approx meters per degree lon/lat near a given latitude (good enough for small extents)."""
    lat = np.deg2rad(lat_deg)
    m_per_deg_lat = 111132.92 - 559.82*np.cos(2*lat) + 1.175*np.cos(4*lat) - 0.0023*np.cos(6*lat)
    m_per_deg_lon = 111412.84*np.cos(lat) - 93.5*np.cos(3*lat) + 0.118*np.cos(5*lat)
    return float(m_per_deg_lon), float(m_per_deg_lat)

def compute_area_dims_from_grid(xmin, xmax, ymin, ymax, dx, dy, assume_points_are_cell_centers=True, units="meters"):
    # range in coordinate units
    width_u  = float(xmax - xmin)
    height_u = float(ymax - ymin)

    # if points are cell centers, add one cell to get edge-to-edge span
    if assume_points_are_cell_centers:
        width_u  += abs(dx)
        height_u += abs(dy)

    if units == "degrees":
        lat_mid = 0.5 * (ymin + ymax)
        m_per_deg_lon, m_per_deg_lat = _meters_per_degree(lat_mid)
        width_m  = width_u  * m_per_deg_lon
        height_m = height_u * m_per_deg_lat
    else:
        width_m  = width_u
        height_m = height_u

    area_m2 = width_m * height_m
    return {
        "width_meters": width_m,
        "width_feet": width_m * FT_PER_M,
        "height_meters": height_m,
        "height_feet": height_m * FT_PER_M,
        "area_m2": area_m2,
        "area_km2": area_m2 / 1e6,
        "area_acres": area_m2 / M2_PER_ACRE,
    }

# =========================
# Example usage (drop-in)
# =========================
df = pd.read_csv(csv_path, low_memory=False)

dx, dy, nx, ny, xmin, xmax, ymin, ymax = estimate_grid_spacing(df, x_col="x", y_col="y")

print(f"Grid dims: {ny} rows  {nx} cols")
print(f"Estimated spacing: dx={dx:.6f}, dy={dy:.6f}")

# --- units heuristic ---
looks_like_degrees = (
    (-180 <= xmin <= 180) and (-180 <= xmax <= 180) and
    (-90  <= ymin <= 90)  and (-90  <= ymax <= 90)
)

units = "degrees" if looks_like_degrees else "meters"
print("Units guess:", "geographic degrees (lon/lat)" if units == "degrees" else "projected (meters)")

area_dims = compute_area_dims_from_grid(
    xmin, xmax, ymin, ymax,
    dx, dy,
    assume_points_are_cell_centers=True,   # set False if x/y are already pixel edges
    units=units
)

# --- your requested outputs ---
print(f"Width (meters):  {area_dims['width_meters']:.2f}")
print(f"Width (feet):    {area_dims['width_feet']:.2f}")
print(f"Height (meters): {area_dims['height_meters']:.2f}")
print(f"Height (feet):   {area_dims['height_feet']:.2f}")

print(f"DEM Area (m):    {area_dims['area_m2']:.2f}")
print(f"DEM Area (km):   {area_dims['area_km2']:.6f}")
print(f"DEM Area (acres): {area_dims['area_acres']:.2f}")


# Summary of site

## Metadata

### Topographic map

In [ ]:
# ============================================================================
# extract meta data from source files
# ============================================================================

import os
import re
import json
import rasterio
import xml.etree.ElementTree as ET

def extract_topo_metadata(tif_path):
    meta = {
        "map_path": tif_path,
        "source": None,
        "year": None,
        "crs": None,
        "bounds": None,
        "width": None,
        "height": None,
        "count": None,
        "dtype": None,
        "driver": None,
        "all_tags": {},
        "sidecar_xml": None,
        "sidecar_txt": None,
    }

    # ------------------------------------------------------------
    # 1) Read raster metadata/tags
    # ------------------------------------------------------------
    with rasterio.open(tif_path) as src:
        meta["crs"] = str(src.crs) if src.crs else None
        meta["bounds"] = tuple(src.bounds)
        meta["width"] = src.width
        meta["height"] = src.height
        meta["count"] = src.count
        meta["dtype"] = src.dtypes[0] if src.count > 0 else None
        meta["driver"] = src.driver

        tags = src.tags()
        band_tags = {f"band_{i}": src.tags(i) for i in range(1, src.count + 1)}
        meta["all_tags"] = {"dataset_tags": tags, "band_tags": band_tags}

        # Search common keys for source/year
        possible_source_keys = [
            "SOURCE", "Source", "source",
            "TIFFTAG_IMAGEDESCRIPTION", "ImageDescription",
            "ORIGIN", "Origin", "origin",
            "PROVIDER", "Provider", "provider"
        ]
        possible_year_keys = [
            "YEAR", "Year", "year",
            "DATE", "Date", "date",
            "ACQDATE", "AcqDate", "acqdate"
        ]

        for k in possible_source_keys:
            if k in tags and tags[k]:
                meta["source"] = tags[k]
                break

        for k in possible_year_keys:
            if k in tags and tags[k]:
                m = re.search(r"(19|20)\d{2}", str(tags[k]))
                if m:
                    meta["year"] = m.group(0)
                    break

    # ------------------------------------------------------------
    # 2) Check sidecar XML
    # ------------------------------------------------------------
    base, _ = os.path.splitext(tif_path)
    xml_path = base + ".xml"
    if os.path.exists(xml_path):
        meta["sidecar_xml"] = xml_path
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            # Flatten text for easy searching
            all_text = " ".join(
                [elem.text.strip() for elem in root.iter() if elem.text and elem.text.strip()]
            )

            if meta["year"] is None:
                m = re.search(r"(19|20)\d{2}", all_text)
                if m:
                    meta["year"] = m.group(0)

            if meta["source"] is None:
                # Look for common USGS / provider phrases
                source_patterns = [
                    r"U\.?S\.? Geological Survey",
                    r"\bUSGS\b",
                    r"\bThe National Map\b",
                    r"\bUS Topo\b",
                    r"\bTopographic Map\b"
                ]
                for pat in source_patterns:
                    m = re.search(pat, all_text, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break

            # Slightly more structured search for citation/origin
            for elem in root.iter():
                tag = elem.tag.lower()
                text = elem.text.strip() if elem.text else ""
                if not text:
                    continue

                if meta["source"] is None and any(x in tag for x in ["origin", "publisher", "source", "citation", "title"]):
                    meta["source"] = text

                if meta["year"] is None and any(x in tag for x in ["date", "pubdate", "year", "timeperd"]):
                    m = re.search(r"(19|20)\d{2}", text)
                    if m:
                        meta["year"] = m.group(0)
        except Exception as e:
            print(f"Warning: could not parse XML: {e}")

    # ------------------------------------------------------------
    # 3) Check sidecar TXT
    # ------------------------------------------------------------
    txt_path = base + ".txt"
    if os.path.exists(txt_path):
        meta["sidecar_txt"] = txt_path
        try:
            with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
                txt = f.read()

            if meta["year"] is None:
                m = re.search(r"(19|20)\d{2}", txt)
                if m:
                    meta["year"] = m.group(0)

            if meta["source"] is None:
                source_patterns = [
                    r"U\.?S\.? Geological Survey",
                    r"\bUSGS\b",
                    r"\bThe National Map\b",
                    r"\bUS Topo\b",
                    r"\bTopographic Map\b"
                ]
                for pat in source_patterns:
                    m = re.search(pat, txt, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break
        except Exception as e:
            print(f"Warning: could not parse TXT: {e}")

    # ------------------------------------------------------------
    # 4) Filename fallback
    # ------------------------------------------------------------
    fname = os.path.basename(tif_path)
    if meta["year"] is None:
        m = re.search(r"(19|20)\d{2}", fname)
        if m:
            meta["year"] = m.group(0)

    if meta["source"] is None:
        if "usgs" in fname.lower():
            meta["source"] = "USGS"
        elif "ustopo" in fname.lower() or "topo" in fname.lower():
            meta["source"] = "Topographic map"

    return meta

In [ ]:
topo_meta = extract_topo_metadata(tif_path)

print(json.dumps(topo_meta, indent=2))
print("Year:", topo_meta["year"])
print("Source:", topo_meta["source"])

### NAIP

In [ ]:
import os
import re
import rasterio
import xml.etree.ElementTree as ET

def extract_naip_metadata(tif_path):
    meta = {
        "path": tif_path,
        "source": None,
        "year": None,
        "acquisition_date": None,
        "crs": None,
        "bounds": None,
        "width": None,
        "height": None,
        "band_count": None,
        "dtypes": None,
        "driver": None,
        "x_res": None,
        "y_res": None,
        "colorinterp": None,
        "dataset_tags": None,
        "band_tags": None,
        "sidecar_xml": None,
        "sidecar_txt": None,
    }

    # ------------------------------------------------------------
    # 1) Read raster metadata
    # ------------------------------------------------------------
    with rasterio.open(tif_path) as src:
        meta["crs"] = str(src.crs) if src.crs else None
        meta["bounds"] = tuple(src.bounds)
        meta["width"] = src.width
        meta["height"] = src.height
        meta["band_count"] = src.count
        meta["dtypes"] = tuple(str(d) for d in src.dtypes)
        meta["driver"] = src.driver
        meta["x_res"] = src.transform.a
        meta["y_res"] = abs(src.transform.e)
        meta["colorinterp"] = tuple(str(c) for c in src.colorinterp)
        meta["dataset_tags"] = src.tags()
        meta["band_tags"] = {f"band_{i}": src.tags(i) for i in range(1, src.count + 1)}

        tags = src.tags()

        # Look for source/provider in tags
        for k in ["SOURCE", "Source", "source", "PROVIDER", "Provider", "provider",
                  "TIFFTAG_IMAGEDESCRIPTION", "ImageDescription", "ORIGIN", "Origin"]:
            if k in tags and tags[k]:
                meta["source"] = tags[k]
                break

        # Look for date/year in tags
        for k in ["ACQDATE", "AcqDate", "acqdate", "DATE", "Date", "date", "YEAR", "Year", "year"]:
            if k in tags and tags[k]:
                txt = str(tags[k])
                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", txt)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["acquisition_date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy
                    break
                m_year = re.search(r"(19|20)\d{2}", txt)
                if m_year:
                    meta["year"] = m_year.group(0)
                    break

    # ------------------------------------------------------------
    # 2) Check sidecar XML
    # ------------------------------------------------------------
    base, _ = os.path.splitext(tif_path)
    xml_path = base + ".xml"
    if os.path.exists(xml_path):
        meta["sidecar_xml"] = xml_path
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            all_text = " ".join(
                [elem.text.strip() for elem in root.iter() if elem.text and elem.text.strip()]
            )

            if meta["source"] is None:
                for pat in [r"\bNAIP\b", r"National Agriculture Imagery Program", r"\bUSDA\b", r"\bUSGS\b"]:
                    m = re.search(pat, all_text, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break

            if meta["acquisition_date"] is None:
                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", all_text)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["acquisition_date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy

            if meta["year"] is None:
                m_year = re.search(r"(19|20)\d{2}", all_text)
                if m_year:
                    meta["year"] = m_year.group(0)

        except Exception as e:
            print(f"Warning: could not parse NAIP XML: {e}")

    # ------------------------------------------------------------
    # 3) Check sidecar TXT
    # ------------------------------------------------------------
    txt_path = base + ".txt"
    if os.path.exists(txt_path):
        meta["sidecar_txt"] = txt_path
        try:
            with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
                txt = f.read()

            if meta["source"] is None:
                for pat in [r"\bNAIP\b", r"National Agriculture Imagery Program", r"\bUSDA\b", r"\bUSGS\b"]:
                    m = re.search(pat, txt, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break

            if meta["acquisition_date"] is None:
                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", txt)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["acquisition_date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy

            if meta["year"] is None:
                m_year = re.search(r"(19|20)\d{2}", txt)
                if m_year:
                    meta["year"] = m_year.group(0)

        except Exception as e:
            print(f"Warning: could not parse NAIP TXT: {e}")

    # ------------------------------------------------------------
    # 4) Filename fallback
    # ------------------------------------------------------------
    fname = os.path.basename(tif_path)

    if meta["acquisition_date"] is None:
        # matches filename endings like 20090716
        m = re.search(r'((19|20)\d{2})(\d{2})(\d{2})', fname)
        if m:
            yyyy, mm, dd = m.group(1), m.group(3), m.group(4)
            meta["acquisition_date"] = f"{yyyy}-{mm}-{dd}"
            meta["year"] = yyyy

    if meta["year"] is None:
        m = re.search(r"(19|20)\d{2}", fname)
        if m:
            meta["year"] = m.group(0)

    if meta["source"] is None:
        if "naip" in fname.lower():
            meta["source"] = "NAIP"
        else:
            meta["source"] = "NAIP imagery"

    return meta

### DEM

In [ ]:
def extract_dem_metadata(tif_path):
    meta = {
        "path": tif_path,
        "source": None,
        "year": None,
        "date": None,
        "crs": None,
        "bounds": None,
        "width": None,
        "height": None,
        "band_count": None,
        "dtypes": None,
        "driver": None,
        "x_res": None,
        "y_res": None,
        "nodata": None,
        "dataset_tags": None,
        "band_tags": None,
        "sidecar_xml": None,
        "sidecar_txt": None,
    }

    # ------------------------------------------------------------
    # 1) Read raster metadata
    # ------------------------------------------------------------
    with rasterio.open(tif_path) as src:
        meta["crs"] = str(src.crs) if src.crs else None
        meta["bounds"] = tuple(src.bounds)
        meta["width"] = src.width
        meta["height"] = src.height
        meta["band_count"] = src.count
        meta["dtypes"] = tuple(str(d) for d in src.dtypes)
        meta["driver"] = src.driver
        meta["x_res"] = src.transform.a
        meta["y_res"] = abs(src.transform.e)
        meta["nodata"] = src.nodata
        meta["dataset_tags"] = src.tags()
        meta["band_tags"] = {f"band_{i}": src.tags(i) for i in range(1, src.count + 1)}

        tags = src.tags()

        # Possible source/provider keys
        for k in [
            "SOURCE", "Source", "source",
            "PROVIDER", "Provider", "provider",
            "ORIGIN", "Origin", "origin",
            "TIFFTAG_IMAGEDESCRIPTION", "ImageDescription"
        ]:
            if k in tags and tags[k]:
                meta["source"] = tags[k]
                break

        # Possible date/year keys
        for k in [
            "DATE", "Date", "date",
            "ACQDATE", "AcqDate", "acqdate",
            "YEAR", "Year", "year"
        ]:
            if k in tags and tags[k]:
                txt = str(tags[k])

                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", txt)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy
                    break

                m_year = re.search(r"(19|20)\d{2}", txt)
                if m_year:
                    meta["year"] = m_year.group(0)
                    break

    # ------------------------------------------------------------
    # 2) Check sidecar XML
    # ------------------------------------------------------------
    base, _ = os.path.splitext(tif_path)
    xml_path = base + ".xml"
    if os.path.exists(xml_path):
        meta["sidecar_xml"] = xml_path
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            all_text = " ".join(
                [elem.text.strip() for elem in root.iter() if elem.text and elem.text.strip()]
            )

            if meta["source"] is None:
                for pat in [
                    r"\bUSGS\b",
                    r"U\.?S\.? Geological Survey",
                    r"\bThe National Map\b",
                    r"\b3DEP\b",
                    r"elevation",
                    r"DEM"
                ]:
                    m = re.search(pat, all_text, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break

            if meta["date"] is None:
                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", all_text)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy

            if meta["year"] is None:
                m_year = re.search(r"(19|20)\d{2}", all_text)
                if m_year:
                    meta["year"] = m_year.group(0)

        except Exception as e:
            print(f"Warning: could not parse DEM XML: {e}")

    # ------------------------------------------------------------
    # 3) Check sidecar TXT
    # ------------------------------------------------------------
    txt_path = base + ".txt"
    if os.path.exists(txt_path):
        meta["sidecar_txt"] = txt_path
        try:
            with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
                txt = f.read()

            if meta["source"] is None:
                for pat in [
                    r"\bUSGS\b",
                    r"U\.?S\.? Geological Survey",
                    r"\bThe National Map\b",
                    r"\b3DEP\b",
                    r"elevation",
                    r"DEM"
                ]:
                    m = re.search(pat, txt, flags=re.IGNORECASE)
                    if m:
                        meta["source"] = m.group(0)
                        break

            if meta["date"] is None:
                m_date = re.search(r"((19|20)\d{2})[-/]?(\d{2})[-/]?(\d{2})", txt)
                if m_date:
                    yyyy, mm, dd = m_date.group(1), m_date.group(3), m_date.group(4)
                    meta["date"] = f"{yyyy}-{mm}-{dd}"
                    meta["year"] = yyyy

            if meta["year"] is None:
                m_year = re.search(r"(19|20)\d{2}", txt)
                if m_year:
                    meta["year"] = m_year.group(0)

        except Exception as e:
            print(f"Warning: could not parse DEM TXT: {e}")

    # ------------------------------------------------------------
    # 4) Filename fallback
    # ------------------------------------------------------------
    fname = os.path.basename(tif_path)

    if meta["date"] is None:
        m = re.search(r'((19|20)\d{2})(\d{2})(\d{2})', fname)
        if m:
            yyyy, mm, dd = m.group(1), m.group(3), m.group(4)
            meta["date"] = f"{yyyy}-{mm}-{dd}"
            meta["year"] = yyyy

    if meta["year"] is None:
        m = re.search(r"(19|20)\d{2}", fname)
        if m:
            meta["year"] = m.group(0)

    if meta["source"] is None:
        lname = fname.lower()
        if "usgs" in lname:
            meta["source"] = "USGS"
        elif "3dep" in lname:
            meta["source"] = "USGS 3DEP"
        elif "dem" in lname:
            meta["source"] = "DEM raster"

    return meta

# Final summary

In [ ]:
# ============================================================================
# DEM SUMMARY STATISTICS (CONSISTENT PRINT/FORMAT)
# ============================================================================

from datetime import datetime

def fmt(x, nd=2):
    """Safe numeric formatter -> string with nd decimals (handles NaN/None)."""
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "NaN"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def fmt_int(x):
    try:
        return f"{int(x)}"
    except Exception:
        return str(x)

def fmt_coord(pt):
    return f"({pt[0]:.6f}, {pt[1]:.6f})"

topo_meta = extract_topo_metadata(topo_tif)
naip_meta = extract_naip_metadata(naip_fp)

tif_path = tif_paths[0] if len(tif_paths) == 1 else os.path.join(path, f"{base}_mosaic.tif")
dem_meta = extract_dem_metadata(tif_path)

slope_col = next(
    (c for c in ["slope_degrees", "slope_degree", "slope_deg", "slope"] if c in df.columns),
    None
)
if slope_col is None:
    lower_cols = {c.lower(): c for c in df.columns}
    slope_col = next(
        (lower_cols[c] for c in ["slope_degrees", "slope_degree", "slope_deg", "slope"] if c in lower_cols),
        None
    )

if slope_col is None:
    print("No slope column found in df; summary slope stats will be NaN.")
    slope_values = pd.Series([np.nan], dtype="float64")
else:
    print("Using slope column for summary:", slope_col)
    slope_values = pd.to_numeric(df[slope_col], errors="coerce")

summary_data = {
    "Site Name": [str(base)],
    "CRS": [str(crs)],
    "DATA_EPSG": [fmt_int(DATA_EPSG)],
    "Resolution dx (map units)": [fmt(dx, 6)],
    "Resolution dy (map units)": [fmt(dy, 6)],
    "Calculated Mean Resolution (m)": [fmt(res_stats["mean_meters"], 2)],
    "Calculated Mean Resolution (ft)": [fmt(res_stats["mean_feet"], 2)],
    "Calculated Median Resolution (m)": [fmt(res_stats["median_meters"], 2)],
    "Calculated Median Resolution (ft)": [fmt(res_stats["median_feet"], 2)],
    "GPS NW Corner": [fmt_coord(NW)],
    "GPS SW Corner": [fmt_coord(SW)],
    "GPS SE Corner": [fmt_coord(SE)],
    "GPS NE Corner": [fmt_coord(NE)],
    "Projection Target EPSG": [fmt_int(DATA_EPSG)],
    "Width (m)": [fmt(area_dims["width_meters"], 2)],
    "Width (ft)": [fmt(area_dims["width_feet"], 2)],
    "Height (m)": [fmt(area_dims["height_meters"], 2)],
    "Height (ft)": [fmt(area_dims["height_feet"], 2)],
    "Max Elevation (m)": [fmt(np.nanmax(Z), 2)],
    "Min Elevation (m)": [fmt(np.nanmin(Z), 2)],
    "Mean Elevation (m)": [fmt(np.nanmean(Z), 2)],
    "Elevation Std Dev (m)": [fmt(np.nanstd(Z), 2)],
    "Mean Slope (deg)": [fmt(slope_values.mean(), 2)],
    "Min Slope (deg)": [fmt(slope_values.min(), 2)],
    "Max Slope (deg)": [fmt(slope_values.max(), 2)],
    "Total Grid Cells": [fmt_int(Z.size)],
    "Grid Rows": [fmt_int(Z.shape[0])],
    "Grid Columns": [fmt_int(Z.shape[1])],
    "DEM Area (m^2)": [fmt(area_dims["area_m2"], 2)],
    "DEM Area (km^2)": [fmt(area_dims["area_km2"], 6)],
    "DEM Area (acres)": [fmt(area_dims["area_acres"], 2)],
    "Processing Date": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    "Source CSV": [str(file_name)],
    "Data Source": [str(data_source)],
    "FinalCSV": [os.path.join(path, f"{base}_slopes.csv")],
    "Source RBG sattelite imagery path": [os.path.join(naip_path, f"{base}.tif")],
    "Source Topomap": [str(topo_tif)],
    "Source DEM imagery path": [", ".join(tif_paths)],
    "Topographic Map Source": [str(topo_meta["source"])],
    "Topographic Map Year": [str(topo_meta["year"])],
    "Topographic Map CRS": [str(topo_meta["crs"])],
    "Topographic Map File": [str(topo_meta["map_path"])],
    "Source RGB satellite imagery path": [naip_meta["path"]],
    "RGB imagery source": [str(naip_meta["source"])],
    "RGB imagery acquisition date": [str(naip_meta["acquisition_date"])],
    "RGB imagery year": [str(naip_meta["year"])],
    "RGB imagery CRS": [str(naip_meta["crs"])],
    "RGB imagery resolution x": [fmt(naip_meta["x_res"], 6)],
    "RGB imagery resolution y": [fmt(naip_meta["y_res"], 6)],
    "RGB imagery width (pixels)": [fmt_int(naip_meta["width"])],
    "RGB imagery height (pixels)": [fmt_int(naip_meta["height"])],
    "RGB imagery band count": [fmt_int(naip_meta["band_count"])],
    "RGB imagery dtype": [str(naip_meta["dtypes"])],
    "RGB imagery colorinterp": [str(naip_meta["colorinterp"])],
    "DEM source": [str(dem_meta["source"])],
    "DEM year": [str(dem_meta["year"])],
    "DEM date": [str(dem_meta["date"])],
    "DEM CRS": [str(dem_meta["crs"])],
    "DEM resolution x": [fmt(dem_meta["x_res"], 6)],
    "DEM resolution y": [fmt(dem_meta["y_res"], 6)],
    "DEM width (pixels)": [fmt_int(dem_meta["width"])],
    "DEM height (pixels)": [fmt_int(dem_meta["height"])],
    "DEM band count": [fmt_int(dem_meta["band_count"])],
    "DEM dtype": [str(dem_meta["dtypes"])],
    "DEM nodata": [str(dem_meta["nodata"])],
}

summary_df = pd.DataFrame(summary_data)
summary_df_T = summary_df.T
summary_df_T.columns = ["Value"]
summary_df_T.index.name = "Parameter"

summary_filename = f"{base}_site_dem_summary.csv"
summary_path = os.path.join(path, summary_filename)
summary_df_T.to_csv(summary_path)

banner = "=" * 72
print("\n" + banner)
print(str(base) + " DEM SUMMARY SAVED")
print(banner)
print(f"Saved summary to: {summary_path}")
print(banner)
print(summary_df_T.to_string())
print(banner)

## Crop a slopes CSV by latitude/longitude bounds
Use this helper to crop an existing *_slopes.csv file with a lat/lon rectangle. It reads lon and lat, keeps rows inside the requested bounds, and writes a new cropped CSV beside the source file.


In [ ]:
"""

from pathlib import Path
import pandas as pd
csv_path = Path(r"C:\Users\jordan.kennedy\Documents\Project\MultiAgent simulation\Datasets\TroyVT\TroyVT_slopes.csv")
out_path = csv_path.with_name("TroyVT_slopes_cropped_troy_vt.csv")
# Troy, VT bounds
NE = (-72.385257, 45.006363)
SW = (-72.392559, 45.000573)
SE = (-72.385257, 45.000573)
NW = (-72.392559, 45.006363)
lon_min = min(SW[0], NW[0])
lon_max = max(NE[0], SE[0])
lat_min = min(SW[1], SE[1])
lat_max = max(NE[1], NW[1])
print(f"Reading: {csv_path}")
df_crop = pd.read_csv(csv_path, low_memory=False)
required_cols = ["lon", "lat"]
missing_cols = [c for c in required_cols if c not in df_crop.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for lat/lon crop: {missing_cols}")
for c in ["lon", "lat"]:
    df_crop[c] = pd.to_numeric(df_crop[c], errors="coerce")
df_crop = df_crop[
    (df_crop["lon"] >= lon_min) & (df_crop["lon"] <= lon_max) &
    (df_crop["lat"] >= lat_min) & (df_crop["lat"] <= lat_max)
].copy()
df_crop.to_csv(out_path, index=False)
print(f"Saved {len(df_crop):,} rows to: {out_path}")
if not df_crop.empty:
    print(df_crop[["lon", "lat"]].agg(["min", "max"]))

"""
